# 🎧 StemLab — separador de voz e instrumentos

Cole um **link do YouTube** (ou envie um arquivo do computador) e a inteligência artificial separa a música em **voz, playback, bateria, baixo, guitarra e piano**. Ouça o resultado em um **mixer** dentro da página e baixe cada faixa em WAV, FLAC ou MP3. Também dá para **baixar só o áudio** do YouTube em vários formatos e qualidades.

**Como usar (2 cliques):**

1. Rode a célula **1️⃣** — faça login na conta Google (obrigatório). Ela também instala o separador e o modelo de IA.
2. Rode a célula **2️⃣** — o aplicativo aparece logo abaixo. Cole o link e siga as etapas.

> 💡 Ou use **Ambiente de execução ▸ Executar tudo** (`Ctrl+F9`).

> ⚡ **Este notebook já abre com GPU (T4) selecionada.** Se o chip no topo do app mostrar *Sem GPU*, vá em **Ambiente de execução ▸ Alterar o tipo de ambiente de execução ▸ T4 GPU** e rode as células de novo.

**Vídeos privados, +18 ou só para membros:** dentro do app, abra a seção *Cookies da sua conta* e carregue o `cookies.txt` exportado do navegador uma única vez. Com o Drive conectado ele fica salvo em `Meu Drive/StemLab/cookies.txt`.

> ⚠️ Uso **pessoal**. Respeite os termos do YouTube e os direitos autorais dos criadores. Nunca compartilhe seu arquivo de cookies.

---
*Desenvolvido por **@emersonms** - 2026*


In [ ]:
#@title 1️⃣ Preparar o ambiente e entrar com a conta Google  { display-mode: "form" }
#@markdown Clique no ▶ à esquerda. Uma janela do Google vai pedir autorização — isso é obrigatório para usar o app. A instalação leva de 1 a 3 minutos.
conectar_google_drive = True  #@param {type:"boolean"}
#@markdown *Com o Drive conectado, os cookies ficam salvos entre sessões e as faixas podem ser copiadas para o Drive.*

import os, sys, shutil, subprocess, json, time
from IPython.display import HTML, display, clear_output

def _box(msg, kind='info'):
    cor = {'info': '#3b82f6', 'ok': '#22c55e', 'warn': '#f59e0b', 'err': '#ef4444'}[kind]
    display(HTML(f'<div style="font-family:system-ui,sans-serif;font-size:14px;padding:10px 14px;margin:6px 0;border-radius:10px;'
                 f'border-left:4px solid {cor};background:{cor}14;color:inherit">{msg}</div>'))

# ---------- 1. Login Google (obrigatório) ----------
_box('🔐 Aguardando login na conta Google… aceite a janela que abriu.')
try:
    from google.colab import auth
    auth.authenticate_user()
except Exception as e:
    raise SystemExit(f'Login cancelado ou falhou: {e}. Rode a célula novamente e conclua o login.')

USER_EMAIL = None
try:
    import google.auth, google.auth.transport.requests, requests
    _creds, _ = google.auth.default()
    _creds.refresh(google.auth.transport.requests.Request())
    USER_EMAIL = requests.get('https://www.googleapis.com/oauth2/v3/userinfo',
                              headers={'Authorization': f'Bearer {_creds.token}'}, timeout=15).json().get('email')
except Exception:
    pass
USER_EMAIL = USER_EMAIL or 'Conta Google conectada'

# ---------- 2. Google Drive (opcional) ----------
DRIVE_OK = False
if conectar_google_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_OK = os.path.isdir('/content/drive/MyDrive')
    except Exception as e:
        _box(f'Drive não conectado: {e}', 'warn')

# ---------- 3. GPU ----------
GPU_NAME = None
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
except Exception:
    pass
if GPU_NAME:
    _box(f'⚡ GPU detectada: <b>{GPU_NAME}</b>', 'ok')
else:
    _box('⚠️ <b>Nenhuma GPU nesta sessão.</b> A separação vai ficar muito lenta. Vá em <b>Ambiente de execução ▸ Alterar o tipo de ambiente de execução ▸ T4 GPU</b> e rode esta célula de novo.', 'warn')

# ---------- 4. Dependências ----------
_box('📦 Instalando yt-dlp, o separador de faixas (audio-separator) e o runtime JavaScript… 1 a 3 min')
_t0 = time.time()
_r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'yt-dlp[default]', 'audio-separator[cpu]', 'audioread'],
                    capture_output=True, text=True)
if _r.returncode != 0:
    _box('Falha ao instalar as dependências. Copie a mensagem abaixo e envie para suporte.', 'err')
    print(_r.stderr[-3000:])
    raise SystemExit('pip falhou')
if not shutil.which('deno'):
    subprocess.run('curl -fsSL https://deno.land/install.sh -o /tmp/deno_install.sh && '
                   'DENO_INSTALL=/usr/local sh /tmp/deno_install.sh < /dev/null', shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not shutil.which('deno') and os.path.exists('/usr/local/bin/deno'):
        os.environ['PATH'] += ':/usr/local/bin'
import importlib, yt_dlp
importlib.reload(yt_dlp)
try:
    import audio_separator
    SEP_VERSION = getattr(audio_separator, '__version__', None) or 'ok'
except Exception as e:
    SEP_VERSION = None

clear_output()
_box(f'✅ <b>Login OK</b> — {USER_EMAIL}', 'ok')
_box(('✅ <b>Google Drive conectado</b> — cookies e faixas podem ser salvos em <code>Meu Drive/StemLab</code>' if DRIVE_OK
      else 'ℹ️ Google Drive <b>não conectado</b> — os cookies valerão só nesta sessão.'), 'ok' if DRIVE_OK else 'info')
_box((f'⚡ GPU: <b>{GPU_NAME}</b>' if GPU_NAME else '⚠️ <b>Sem GPU</b> — ative em Ambiente de execução ▸ Alterar o tipo de ambiente de execução ▸ T4 GPU'), 'ok' if GPU_NAME else 'warn')
_box(f'✅ yt-dlp <b>{yt_dlp.version.__version__}</b> · separador {"<b>instalado</b>" if SEP_VERSION else "<b>NÃO instalado</b>"} · ffmpeg {"OK" if shutil.which("ffmpeg") else "NÃO encontrado"} · '
     f'deno {"OK" if shutil.which("deno") else "não instalado"} · instalação em {int(time.time()-_t0)} s', 'ok' if SEP_VERSION else 'err')
_box('👉 Agora rode a célula <b>2️⃣ Abrir o aplicativo</b>. O modelo de IA principal começa a ser baixado em segundo plano assim que o app abrir.', 'info')


In [ ]:
#@title 2️⃣ Abrir o aplicativo  { display-mode: "form" }
#@markdown Clique no ▶. A interface aparece logo abaixo. Se o ambiente for reiniciado, rode a célula 1 e depois esta.
import os as _os
from IPython.display import HTML as _HTML, display as _display

def _erro(msg):
    _display(_HTML(f'<div style="font-family:system-ui,sans-serif;padding:14px 16px;border-radius:12px;border-left:4px solid #ef4444;'
                   f'background:#ef444414;font-size:14px">⛔ <b>{msg}</b></div>'))

_pronto = True
if 'USER_EMAIL' not in globals():
    _erro('Execute primeiro a célula 1️⃣ (login na conta Google).'); _pronto = False
else:
    try:
        import google.auth as _ga; _ga.default()
    except Exception:
        _erro('Login na conta Google necessário. Rode a célula 1️⃣ novamente.'); _pronto = False
    try:
        import audio_separator as _as
    except Exception:
        _erro('O separador de faixas não está instalado. Rode a célula 1️⃣ novamente e aguarde a instalação terminar.'); _pronto = False

if _pronto:
    try:
        # ============================================================
        #  StemLab — backend (roda dentro do Google Colab)
        #  Expõe funções Python para a interface web via
        #  google.colab.output.register_callback / kernel.invokeFunction
        # ============================================================
        import os, re, json, time, uuid, shutil, threading, socketserver, http.server
        import urllib.parse, traceback, queue, subprocess, base64, sys

        import yt_dlp
        from IPython.display import JSON

        # ---------- caminhos ----------
        BASE_DIR      = '/content/stemlab'
        WORK_DIR      = os.path.join(BASE_DIR, 'work')       # áudio de origem (wav) e uploads
        OUT_DIR       = os.path.join(BASE_DIR, 'out')        # resultados por tarefa (servidos ao navegador)
        MODELS_DIR    = os.path.join(BASE_DIR, 'models')     # modelos de IA baixados
        COOKIES_PATH  = os.path.join(BASE_DIR, 'cookies.txt')
        DRIVE_MYDRIVE = '/content/drive/MyDrive'
        DRIVE_ROOT    = os.path.join(DRIVE_MYDRIVE, 'StemLab')
        FILE_PORT     = 8766
        MAX_UPLOAD_MB = 250
        MAX_DURATION  = 25 * 60   # 25 min: acima disso a GPU gratuita costuma estourar a memória
        MOCK          = bool(os.environ.get('STEMLAB_MOCK'))  # modo de teste local sem GPU/modelos
        for _d in (WORK_DIR, OUT_DIR, MODELS_DIR):
            os.makedirs(_d, exist_ok=True)

        STATE = {
            'jobs': {}, 'sources': {}, 'uploads': {},
            'user': globals().get('USER_EMAIL') or 'Conta Google conectada',
            'js_runtime': None, 'gpu': None, 'prefetch': {},
        }

        # ---------- modos de separação ----------
        STEM_PT = {'Vocals': 'Voz', 'Instrumental': 'Playback', 'Drums': 'Bateria', 'Bass': 'Baixo',
                   'Guitar': 'Guitarra', 'Piano': 'Piano', 'Other': 'Outros'}
        STEM_ICON = {'Vocals': '🎤', 'Instrumental': '🎹', 'Drums': '🥁', 'Bass': '🎸', 'Guitar': '🎸', 'Piano': '🎹', 'Other': '🎼'}
        STEM_COLOR = {'Vocals': '#f472b6', 'Instrumental': '#38bdf8', 'Drums': '#fb923c', 'Bass': '#a78bfa',
                      'Guitar': '#facc15', 'Piano': '#34d399', 'Other': '#94a3b8'}
        MODES = {
            '2stem': {'label': 'Voz + Playback', 'model': 'model_bs_roformer_ep_317_sdr_12.9755.ckpt',
                      'stems': ['Vocals', 'Instrumental'], 'factor': 1.2, 'base': 30},
            '4stem': {'label': 'Banda (4 faixas)', 'model': 'htdemucs_ft.yaml',
                      'stems': ['Vocals', 'Drums', 'Bass', 'Other'], 'factor': 1.2, 'base': 40},
            '6stem': {'label': 'Completo (6 faixas)', 'model': 'htdemucs_6s.yaml',
                      'stems': ['Vocals', 'Drums', 'Bass', 'Guitar', 'Piano', 'Other'], 'factor': 0.5, 'base': 30},
        }
        OUT_FORMATS = {'wav': 'WAV', 'flac': 'FLAC', 'mp3': 'MP3'}

        # ---------- utilidades ----------
        def _drive_on():
            return os.path.isdir(DRIVE_MYDRIVE)

        def _detect_js_runtime():
            if STATE['js_runtime'] is not None:
                return STATE['js_runtime']
            rt = {}
            for name in ('deno', 'node', 'bun'):
                p = shutil.which(name)
                if p:
                    rt = {name: {'path': p}}
                    break
            STATE['js_runtime'] = rt
            return rt

        def _gpu_info(force=False):
            if STATE['gpu'] is not None and not force:
                return STATE['gpu']
            info = {'available': False, 'name': None, 'memory_gb': None, 'mock': MOCK}
            try:
                import torch
                if torch.cuda.is_available():
                    p = torch.cuda.get_device_properties(0)
                    info.update({'available': True, 'name': p.name, 'memory_gb': round(p.total_memory / 1024 ** 3, 1)})
            except Exception:
                pass
            STATE['gpu'] = info
            return info

        def _friendly_error(msg):
            m = (msg or '').lower()
            if 'sign in to confirm' in m or 'not a bot' in m or ('bot' in m and 'confirm' in m):
                return ('O YouTube pediu confirmação de que você não é um robô. Isso é comum em servidores do Colab. '
                        'Carregue os cookies da sua conta na seção "Cookies" e tente novamente.')
            if 'private video' in m or 'this video is private' in m:
                return 'Vídeo privado. Carregue os cookies de uma conta com acesso para usá-lo.'
            if 'members-only' in m or 'join this channel' in m:
                return 'Conteúdo exclusivo para membros. Carregue os cookies da conta que é membro do canal.'
            if 'age' in m and ('restrict' in m or 'confirm your age' in m or 'inappropriate' in m):
                return 'Vídeo com restrição de idade. Carregue os cookies da sua conta para confirmar a idade.'
            if 'video unavailable' in m or 'video is unavailable' in m or 'not available' in m:
                return 'Vídeo indisponível (removido, bloqueado na região ou link inválido).'
            if 'unsupported url' in m or 'is not a valid url' in m:
                return 'Este link não é reconhecido. Cole um link de vídeo do YouTube.'
            if 'requested format is not available' in m:
                return 'O formato escolhido não está disponível para este vídeo. Tente outro.'
            if 'http error 429' in m or 'too many requests' in m:
                return 'O YouTube limitou as requisições (429). Aguarde alguns minutos ou use cookies.'
            if 'live' in m and ('is a live' in m or 'not yet' in m or 'premiere' in m):
                return 'Transmissões ao vivo ou estreias ainda não disponíveis não podem ser processadas.'
            if 'out of memory' in m or 'cuda error' in m:
                return 'A memória da GPU estourou. Tente uma música mais curta ou o modo "Voz + Playback".'
            return re.sub(r'^\s*ERROR:\s*', '', msg or 'Erro desconhecido').strip()[:400]

        class _SilentLogger:
            """Engole a saída do yt-dlp, mas guarda erros para reportar na interface."""
            def __init__(self): self.errors, self.warnings = [], []
            def debug(self, msg):   pass
            def info(self, msg):    pass
            def warning(self, msg): self.warnings.append(str(msg))
            def error(self, msg):   self.errors.append(str(msg))

        def _base_opts():
            o = {
                'quiet': True, 'no_warnings': True, 'noprogress': True, 'nocheckcertificate': True,
                'extractor_retries': 3, 'retries': 5, 'fragment_retries': 5, 'socket_timeout': 30,
                'geo_bypass': True, 'logger': _SilentLogger(),
            }
            if os.path.exists(COOKIES_PATH):
                o['cookiefile'] = COOKIES_PATH
            rt = _detect_js_runtime()
            if rt:
                o['js_runtimes'] = rt
            return o

        def _fmt_size(b):
            if not b: return None
            for unit in ('B', 'KB', 'MB', 'GB'):
                if b < 1024: return f'{b:.0f} {unit}' if unit == 'B' else f'{b:.1f} {unit}'
                b /= 1024
            return f'{b:.1f} TB'

        def _hms(sec):
            sec = int(round(sec or 0)); h, m, s = sec // 3600, sec % 3600 // 60, sec % 60
            return f'{h}:{m:02d}:{s:02d}' if h else f'{m}:{s:02d}'

        def _thumb(info):
            t = info.get('thumbnail')
            if not t and info.get('thumbnails'):
                t = info['thumbnails'][-1].get('url')
            if not t and info.get('id'):
                t = f"https://i.ytimg.com/vi/{info['id']}/hqdefault.jpg"
            return t

        def _safe_name(s, n=90):
            s = yt_dlp.utils.sanitize_filename(str(s or 'audio'), restricted=False)
            s = re.sub(r'\s+', ' ', s).strip(' .')
            return s[:n] or 'audio'

        def _parse_video_id(url):
            u = url.strip()
            if not re.match(r'^https?://', u, re.I):
                u = 'https://' + u
            p = urllib.parse.urlparse(u)
            host = p.netloc.lower().replace('www.', '').replace('m.', '').replace('music.', '')
            qs = urllib.parse.parse_qs(p.query)
            if host == 'youtu.be':
                vid = p.path.strip('/').split('/')[0] or None
            elif 'youtube' in host:
                vid = (qs.get('v') or [None])[0]
                if not vid:
                    m = re.match(r'^/(shorts|live|embed|v)/([A-Za-z0-9_-]{11})', p.path)
                    vid = m.group(2) if m else None
            else:
                raise ValueError('unsupported url: apenas links do YouTube são aceitos.')
            if not vid:
                if (qs.get('list') or [None])[0]:
                    raise ValueError('Este link é de uma playlist. O StemLab trabalha com uma música por vez: abra o vídeo e copie o link dele.')
                raise ValueError('unsupported url: não encontrei um vídeo nesse link.')
            return vid

        def _ffprobe_duration(path):
            try:
                out = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', path],
                                     capture_output=True, text=True, timeout=60).stdout.strip()
                return float(out) if out else 0.0
            except Exception:
                return 0.0

        # ---------- cookies ----------
        def _json_cookies_to_netscape(items):
            lines = ['# Netscape HTTP Cookie File', '# gerado pelo StemLab', '']
            for c in items:
                dom = c.get('domain') or '.youtube.com'
                flag = 'TRUE' if dom.startswith('.') else 'FALSE'
                path = c.get('path') or '/'
                secure = 'TRUE' if c.get('secure') else 'FALSE'
                exp = c.get('expirationDate') or c.get('expires') or c.get('expiry') or 0
                try: exp = int(float(exp))
                except Exception: exp = 0
                lines.append('\t'.join([dom, flag, path, secure, str(exp), str(c.get('name', '')), str(c.get('value', ''))]))
            return '\n'.join(lines) + '\n'

        def _header_cookies_to_netscape(header):
            header = re.sub(r'^\s*cookie\s*:\s*', '', header.strip(), flags=re.I)
            items = []
            for part in header.split(';'):
                if '=' in part:
                    n, v = part.strip().split('=', 1)
                    items.append({'domain': '.youtube.com', 'path': '/', 'secure': True,
                                  'expirationDate': int(time.time()) + 365 * 86400, 'name': n.strip(), 'value': v.strip()})
            return _json_cookies_to_netscape(items)

        def _normalize_cookies(text):
            t = text.strip().lstrip('﻿')
            if not t:
                raise ValueError('Arquivo de cookies vazio.')
            if t.startswith('[') or t.startswith('{'):
                data = json.loads(t)
                if isinstance(data, dict):
                    data = data.get('cookies') or data.get('items') or list(data.values())
                return _json_cookies_to_netscape(data)
            if '\t' in t:
                if not t.startswith('#'):
                    t = '# Netscape HTTP Cookie File\n' + t
                return t + '\n'
            if '=' in t and ';' in t and '\n' not in t.strip():
                return _header_cookies_to_netscape(t)
            raise ValueError('Formato de cookies não reconhecido. Use um arquivo cookies.txt (formato Netscape) ou JSON exportado por extensão.')

        def _cookie_summary():
            if not os.path.exists(COOKIES_PATH):
                return {'found': False}
            names, total = set(), 0
            with open(COOKIES_PATH, encoding='utf-8', errors='ignore') as f:
                for line in f:
                    if line.startswith('#') or not line.strip(): continue
                    parts = line.rstrip('\n').split('\t')
                    if len(parts) >= 7 and ('youtube.com' in parts[0] or 'google.com' in parts[0]):
                        total += 1
                        names.add(parts[5])
            logged = bool(names & {'SID', '__Secure-3PSID', 'SAPISID', '__Secure-3PAPISID', 'LOGIN_INFO'})
            src = 'drive' if os.path.exists(os.path.join(DRIVE_ROOT, 'cookies.txt')) else 'sessao'
            return {'found': True, 'total': total, 'logged_in': logged, 'source': src,
                    'updated': time.strftime('%d/%m/%Y %H:%M', time.localtime(os.path.getmtime(COOKIES_PATH)))}

        def cb_save_cookies(text, persist_drive=True):
            try:
                content = _normalize_cookies(text)
                with open(COOKIES_PATH, 'w', encoding='utf-8') as f:
                    f.write(content)
                saved_drive = False
                if persist_drive and _drive_on():
                    os.makedirs(DRIVE_ROOT, exist_ok=True)
                    shutil.copy(COOKIES_PATH, os.path.join(DRIVE_ROOT, 'cookies.txt'))
                    saved_drive = True
                s = _cookie_summary(); s['saved_drive'] = saved_drive
                return JSON({'ok': True, 'cookies': s})
            except Exception as e:
                return JSON({'ok': False, 'error': _friendly_error(str(e))})

        def cb_clear_cookies(also_drive=False):
            try:
                if os.path.exists(COOKIES_PATH): os.remove(COOKIES_PATH)
                if also_drive:
                    p = os.path.join(DRIVE_ROOT, 'cookies.txt')
                    if os.path.exists(p): os.remove(p)
                return JSON({'ok': True, 'cookies': _cookie_summary()})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_test_cookies():
            try:
                if not os.path.exists(COOKIES_PATH):
                    return JSON({'ok': False, 'error': 'Nenhum cookie carregado.'})
                opts = _base_opts(); opts.update({'extract_flat': True, 'playlistend': 1, 'ignoreerrors': True})
                with yt_dlp.YoutubeDL(opts) as ydl:
                    info = ydl.extract_info('https://www.youtube.com/feed/history', download=False)
                ok = bool(info and (info.get('entries') is not None))
                return JSON({'ok': True, 'valid': ok,
                             'message': 'Cookies válidos: sua conta foi reconhecida pelo YouTube.' if ok
                             else 'O YouTube não reconheceu a sessão. Exporte os cookies novamente com a conta logada.'})
            except Exception as e:
                return JSON({'ok': True, 'valid': False, 'message': _friendly_error(str(e))})

        # ---------- bootstrap ----------
        def cb_bootstrap():
            try:
                if not os.path.exists(COOKIES_PATH) and _drive_on():
                    saved = os.path.join(DRIVE_ROOT, 'cookies.txt')
                    if os.path.exists(saved):
                        shutil.copy(saved, COOKIES_PATH)
                return JSON({'ok': True, 'user': STATE['user'], 'drive': _drive_on(), 'cookies': _cookie_summary(),
                             'ytdlp': yt_dlp.version.__version__, 'js_runtime': next(iter(_detect_js_runtime()), None),
                             'gpu': _gpu_info(), 'port': FILE_PORT, 'mock': MOCK, 'max_upload_mb': MAX_UPLOAD_MB,
                             'max_duration': MAX_DURATION,
                             'modes': {k: {'label': v['label'], 'stems': v['stems'], 'factor': v['factor'], 'base': v['base'],
                                           'ready': _model_ready(v['model'])} for k, v in MODES.items()},
                             'stem_pt': STEM_PT, 'stem_icon': STEM_ICON, 'stem_color': STEM_COLOR})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        # ---------- análise de link ----------
        def _summarize_audio_formats(info):
            auds, seen = [], set()
            duration = info.get('duration') or 0
            for f in info.get('formats') or []:
                vcodec, acodec = f.get('vcodec') or 'none', f.get('acodec') or 'none'
                if acodec == 'none' or vcodec != 'none':
                    continue
                if 'drc' in (f.get('format_id') or '') or 'DRC' in (f.get('format_note') or ''):
                    continue
                size = f.get('filesize') or f.get('filesize_approx') or 0
                if not size and duration and f.get('tbr'):
                    size = int(f['tbr'] * 1000 / 8 * duration)
                abr = int(round(f.get('abr') or f.get('tbr') or 0))
                key = (f.get('ext'), acodec.split('.')[0], abr // 8)
                if key in seen: continue
                seen.add(key)
                auds.append({'ext': f.get('ext'), 'acodec': acodec.split('.')[0], 'abr': abr, 'size': size, 'size_label': _fmt_size(size)})
            auds.sort(key=lambda a: -a['abr'])
            return auds

        def cb_analyze(url):
            try:
                vid = _parse_video_id(url)
                opts = _base_opts(); opts['noplaylist'] = True
                with yt_dlp.YoutubeDL(opts) as ydl:
                    info = ydl.extract_info(f'https://www.youtube.com/watch?v={vid}', download=False)
                if not info:
                    raise RuntimeError('video unavailable')
                if info.get('is_live'):
                    raise RuntimeError('is a live')
                dur = info.get('duration') or 0
                src_id = uuid.uuid4().hex[:10]
                src = {
                    'id': src_id, 'kind': 'youtube', 'video_id': info.get('id'), 'title': info.get('title'),
                    'channel': info.get('uploader') or info.get('channel'), 'duration': dur, 'duration_str': _hms(dur),
                    'views': info.get('view_count'), 'upload_date': info.get('upload_date'), 'thumbnail': _thumb(info),
                    'url': info.get('webpage_url') or f'https://www.youtube.com/watch?v={vid}',
                    'audio_formats': _summarize_audio_formats(info), 'wav': None, 'created': time.time(),
                    'too_long': dur > MAX_DURATION,
                }
                STATE['sources'][src_id] = src
                return JSON({'ok': True, 'source': src})
            except Exception as e:
                return JSON({'ok': False, 'error': _friendly_error(str(e))})

        # ---------- upload de arquivo local (em pedaços base64) ----------
        def cb_upload_start(name, size):
            try:
                size = int(size or 0)
                if size <= 0:
                    raise ValueError('Arquivo vazio.')
                if size > MAX_UPLOAD_MB * 1024 * 1024:
                    raise ValueError(f'O arquivo tem {_fmt_size(size)}. O limite é {MAX_UPLOAD_MB} MB.')
                ext = os.path.splitext(name or '')[1].lower()
                if ext not in ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg', '.opus', '.wma', '.aiff', '.aif', '.mp4', '.webm', '.mkv', '.mov'):
                    raise ValueError('Formato não suportado. Envie MP3, WAV, FLAC, M4A, OGG, OPUS, AIFF ou um vídeo MP4/WEBM.')
                up_id = uuid.uuid4().hex[:10]
                path = os.path.join(WORK_DIR, f'upload_{up_id}{ext}')
                open(path, 'wb').close()
                STATE['uploads'][up_id] = {'path': path, 'name': name, 'size': size, 'received': 0}
                return JSON({'ok': True, 'upload_id': up_id})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_upload_chunk(up_id, data_b64):
            try:
                up = STATE['uploads'].get(up_id)
                if not up: raise ValueError('Envio não encontrado. Tente novamente.')
                chunk = base64.b64decode(data_b64)
                with open(up['path'], 'ab') as f:
                    f.write(chunk)
                up['received'] += len(chunk)
                if up['received'] > MAX_UPLOAD_MB * 1024 * 1024:
                    raise ValueError('Arquivo maior que o limite.')
                return JSON({'ok': True, 'received': up['received']})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_upload_finish(up_id):
            try:
                up = STATE['uploads'].pop(up_id, None)
                if not up: raise ValueError('Envio não encontrado. Tente novamente.')
                if up['received'] != up['size']:
                    raise ValueError(f'Envio incompleto ({_fmt_size(up["received"])} de {_fmt_size(up["size"])}). Tente novamente.')
                dur = _ffprobe_duration(up['path'])
                if dur <= 0:
                    os.remove(up['path'])
                    raise ValueError('Não consegui ler este arquivo como áudio. Verifique se ele não está corrompido.')
                src_id = uuid.uuid4().hex[:10]
                title = os.path.splitext(os.path.basename(up['name']))[0]
                src = {'id': src_id, 'kind': 'file', 'title': title, 'channel': None, 'duration': dur, 'duration_str': _hms(dur),
                       'file_name': up['name'], 'file_size': up['size'], 'file_size_label': _fmt_size(up['size']),
                       'thumbnail': None, 'url': None, 'audio_formats': [], 'wav': None, 'src_path': up['path'],
                       'created': time.time(), 'too_long': dur > MAX_DURATION}
                STATE['sources'][src_id] = src
                return JSON({'ok': True, 'source': src})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        # ---------- tarefas ----------
        JOB_QUEUE = queue.Queue()

        def _stage(job, key, label, status='running', percent=None):
            for s in job['stages']:
                if s['key'] == key:
                    s['status'] = status
                    if percent is not None: s['percent'] = percent
                    if status == 'running':
                        s['started'] = s.get('started') or time.time()
                    if status in ('done', 'error'):
                        s['ended'] = time.time()
                elif s['status'] == 'running' and status == 'running':
                    s['status'] = 'done'; s['percent'] = 100; s['ended'] = time.time()
            if job.get('stage') != key: job['detail'] = ''
            job['stage'] = key; job['stage_label'] = label
            if percent is not None: job['percent'] = percent
            if status == 'running': job['status'] = 'running'

        def _check_cancel(job):
            if job.get('cancel'):
                raise yt_dlp.utils.DownloadCancelled('Cancelado pelo usuário')

        def _yt_progress(d, job):
            _check_cancel(job)
            if d['status'] == 'downloading':
                total = d.get('total_bytes') or d.get('total_bytes_estimate') or 0
                done = d.get('downloaded_bytes') or 0
                job['percent'] = (done / total * 100) if total else 0
                job['detail'] = f"{_fmt_size(done) or '0 B'}{' de ' + _fmt_size(total) if total else ''}" + (f" · {_fmt_size(d['speed'])}/s" if d.get('speed') else '')
            elif d['status'] == 'finished':
                job['percent'] = 100; job['detail'] = ''

        def _final_path(info, out_dir, before):
            rd = (info or {}).get('requested_downloads') or []
            if rd and rd[0].get('filepath') and os.path.exists(rd[0]['filepath']):
                return rd[0]['filepath']
            cands = [os.path.join(out_dir, f) for f in os.listdir(out_dir)
                     if f not in before and not f.endswith(('.part', '.ytdl', '.webp', '.jpg', '.png'))]
            return max(cands, key=os.path.getmtime) if cands else None

        def _download_youtube_audio(job, src, out_dir, audio_format='best', audio_quality='best', embed_meta=True, tag='src'):
            """Baixa o áudio de um vídeo. audio_format 'best' = original (m4a/webm) sem conversão."""
            opts = _base_opts()
            logger = _SilentLogger(); opts['logger'] = logger
            opts.update({
                'outtmpl': os.path.join(out_dir, f'{tag}_%(id)s.%(ext)s' if tag == 'src' else '%(title).120B [%(id)s].%(ext)s'),
                'noplaylist': True, 'overwrites': True, 'windowsfilenames': True, 'concurrent_fragment_downloads': 4,
                'format': 'ba/b', 'progress_hooks': [lambda d: _yt_progress(d, job)], 'postprocessors': [], 'ignoreerrors': True,
                'postprocessor_hooks': [lambda d: _pp_hook(d, job)],
            })
            if audio_format != 'best':
                pp = {'key': 'FFmpegExtractAudio', 'preferredcodec': audio_format}
                if audio_format not in ('wav', 'flac') and audio_quality and audio_quality != 'best':
                    pp['preferredquality'] = str(audio_quality)
                elif audio_format == 'mp3':
                    pp['preferredquality'] = '0'
                opts['postprocessors'].append(pp)
                if embed_meta:
                    opts['postprocessors'].append({'key': 'FFmpegMetadata', 'add_metadata': True})
                    if audio_format in ('mp3', 'm4a', 'flac', 'opus', 'aac'):
                        opts['writethumbnail'] = True
                        opts['postprocessors'].append({'key': 'EmbedThumbnail', 'already_have_thumbnail': False})
            before = set(os.listdir(out_dir))
            with yt_dlp.YoutubeDL(opts) as ydl:
                info = ydl.extract_info(src['url'], download=True)
            _check_cancel(job)
            path = _final_path(info, out_dir, before)
            if not path:
                raise RuntimeError(logger.errors[-1] if logger.errors else 'O arquivo não foi encontrado após o download.')
            if logger.errors:
                job['warning'] = _friendly_error(logger.errors[-1])
            return path

        def _pp_hook(d, job):
            _check_cancel(job)
            names = {'FFmpegExtractAudio': 'Convertendo áudio…', 'FFmpegMetadata': 'Gravando metadados…',
                     'EmbedThumbnail': 'Inserindo capa…', 'MoveFiles': 'Finalizando…'}
            if d['status'] in ('started', 'processing'):
                job['detail'] = names.get(d.get('postprocessor'), 'Processando…')

        def _ensure_source_wav(job, src):
            """Garante um WAV 44.1 kHz estéreo da fonte (baixando do YouTube se preciso). Fica em cache por fonte."""
            if src.get('wav') and os.path.exists(src['wav']):
                _stage(job, 'fetch', 'Áudio já disponível', 'done', 100)
                return src['wav']
            src_dir = os.path.join(WORK_DIR, src['id']); os.makedirs(src_dir, exist_ok=True)
            if src['kind'] == 'youtube':
                _stage(job, 'fetch', 'Baixando áudio do YouTube…', 'running', 0)
                raw = _download_youtube_audio(job, src, src_dir, 'best', tag='src')
            else:
                raw = src['src_path']
            _stage(job, 'fetch', 'Preparando o áudio…', 'running', 100)
            job['detail'] = 'Convertendo para WAV 44.1 kHz'
            wav = os.path.join(src_dir, 'source.wav')
            r = subprocess.run(['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', raw, '-vn', '-ac', '2', '-ar', '44100',
                                '-c:a', 'pcm_s16le', wav], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.exists(wav):
                raise RuntimeError('Falha ao converter o áudio: ' + (r.stderr.strip().splitlines() or ['ffmpeg'])[-1])
            src['wav'] = wav
            if not src.get('duration'):
                src['duration'] = _ffprobe_duration(wav); src['duration_str'] = _hms(src['duration'])
            job['detail'] = ''
            return wav

        # ---- separação (subprocesso cancelável) ----
        def _sep_cmd():
            exe = shutil.which('audio-separator')
            return [exe] if exe else [sys.executable, '-m', 'audio_separator.utils.cli']

        def _model_ready(model):
            return any(f.startswith(os.path.splitext(model)[0]) for f in os.listdir(MODELS_DIR)) if os.path.isdir(MODELS_DIR) else False

        def _iter_output(stream):
            """Le a saida do processo separando por quebra de linha e tambem por retorno de carro (barras do tqdm)."""
            buf = ''
            while True:
                ch = stream.read(1)
                if not ch:
                    if buf: yield buf
                    return
                if ch == chr(13) or ch == chr(10):
                    if buf.strip(): yield buf
                    buf = ''
                else:
                    buf += ch

        def _run_separator(job, wav, mode, out_format, out_dir, names):
            cfg = MODES[mode]
            cmd = _sep_cmd() + [wav, '--model_filename', cfg['model'],
                   '--model_file_dir', MODELS_DIR, '--output_dir', out_dir, '--output_format', OUT_FORMATS[out_format],
                   '--custom_output_names', json.dumps(names), '--log_level', 'info']
            if out_format == 'mp3':
                cmd += ['--output_bitrate', '320k']
            if cfg['model'].endswith('.yaml'):
                cmd += ['--demucs_shifts', '1']
            if _gpu_info()['available']:
                cmd += ['--use_autocast']   # precisão mista na GPU: bem mais rápido, qualidade praticamente igual
            env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8', TQDM_MININTERVAL='0.5')
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='ignore', env=env, bufsize=0)
            job['proc'] = proc
            tail, err_hint = [], None
            started = time.time(); phase_started = None
            est = cfg['base'] + (job.get('duration') or 180) * cfg['factor']
            onde = 'na GPU' if (_gpu_info()['available']) else 'no processador'
            if _model_ready(cfg['model']):
                _stage(job, 'model', f'Carregando o modelo {onde}…', 'running', 0)
            else:
                _stage(job, 'model', 'Baixando o modelo de IA (uma vez por sessão)…', 'running', 0)
            try:
                for line in _iter_output(proc.stdout):
                    if job.get('cancel'):
                        proc.kill(); raise yt_dlp.utils.DownloadCancelled('Cancelado pelo usuário')
                    tail.append(line[:300]); tail[:] = tail[-40:]
                    low = line.lower()
                    m = re.search(r'(\d{1,3})%\|', line)  # barras do tqdm (download do modelo / separação)
                    if 'starting separation' in low:
                        _stage(job, 'separate', f'Separando as faixas {onde}…', 'running', 0); phase_started = time.time(); job['detail'] = ''
                    elif 'loading model' in low and job['stage'] == 'model':
                        job['detail'] = 'Carregando pesos…'
                    elif ' saving ' in low or 'writing output' in low:
                        if job['stage'] == 'separate': job['detail'] = 'Gravando arquivos…'
                    elif m and job['stage'] == 'model':
                        job['percent'] = int(m.group(1)); job['detail'] = f'Baixando modelo… {m.group(1)}%'
                    elif m and job['stage'] == 'separate':
                        job['percent'] = max(job['percent'], min(99, int(m.group(1)))); job['detail'] = f'{m.group(1)}%'
                    if ('error' in low or 'traceback' in low) and 'errorlevel' not in low:
                        err_hint = line
                    if job['stage'] == 'separate' and phase_started and not m:
                        job['percent'] = max(job['percent'], min(95, (time.time() - phase_started) / max(est, 1) * 100))
                proc.wait()
            finally:
                job['proc'] = None
                try: proc.stdout.close()
                except Exception: pass
            if job.get('cancel'):
                raise yt_dlp.utils.DownloadCancelled('Cancelado pelo usuário')
            if proc.returncode != 0:
                msg = err_hint or (tail[-1] if tail else f'o separador retornou código {proc.returncode}')
                job['log_tail'] = tail[-15:]
                raise RuntimeError(_friendly_error(msg))
            job['sep_seconds'] = round(time.time() - started)

        def _run_separator_mock(job, wav, mode, out_format, out_dir, names):
            """Modo de teste local: gera 'faixas' com filtros do ffmpeg, sem IA."""
            filters = {'Vocals': 'highpass=f=1200', 'Instrumental': 'lowpass=f=1200', 'Drums': 'lowpass=f=200',
                       'Bass': 'lowpass=f=120', 'Guitar': 'bandpass=f=800:w=400', 'Piano': 'bandpass=f=1500:w=600', 'Other': 'highpass=f=3000'}
            _stage(job, 'model', 'Carregando o modelo (simulação)…', 'running', 0); time.sleep(1.2)
            _stage(job, 'separate', 'Separando as faixas (simulação)…', 'running', 0)
            stems = MODES[mode]['stems']
            for i, stem in enumerate(stems):
                _check_cancel(job)
                ext = out_format
                dst = os.path.join(out_dir, f'{names[stem]}.{ext}')
                codec = ['-c:a', 'libmp3lame', '-b:a', '320k'] if ext == 'mp3' else (['-c:a', 'flac'] if ext == 'flac' else ['-c:a', 'pcm_s16le'])
                subprocess.run(['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', wav, '-af', filters[stem]] + codec + [dst], check=True)
                job['percent'] = (i + 1) / len(stems) * 100; time.sleep(0.6)
            job['sep_seconds'] = 3

        def _make_preview(src_path, dst_path):
            subprocess.run(['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', src_path, '-vn', '-ac', '2', '-ar', '44100',
                            '-c:a', 'libmp3lame', '-b:a', '128k', dst_path], capture_output=True)
            return os.path.exists(dst_path)

        def _copy_to_drive(job, path, sub):
            if not (job.get('save_drive') and _drive_on()):
                return None
            d = os.path.join(DRIVE_ROOT, sub); os.makedirs(d, exist_ok=True)
            dst = os.path.join(d, os.path.basename(path))
            shutil.copy2(path, dst)
            return dst.replace(DRIVE_MYDRIVE, 'Meu Drive')

        def _run_separate_job(job):
            src = STATE['sources'][job['source_id']]
            job['duration'] = src.get('duration') or 0
            out_dir = os.path.join(OUT_DIR, job['id']); os.makedirs(out_dir, exist_ok=True)
            wav = _ensure_source_wav(job, src)
            if not job['duration']:
                job['duration'] = _ffprobe_duration(wav)
            if job['duration'] > MAX_DURATION:
                raise RuntimeError(f'Este áudio tem {_hms(job["duration"])}. O limite é {MAX_DURATION // 60} minutos para não estourar a memória da GPU.')
            mode, fmt = job['mode'], job['out_format']
            base = _safe_name(src['title'])
            names = {stem: f'stemlab_{stem.lower()}' for stem in MODES[mode]['stems']}   # nomes internos simples
            (_run_separator_mock if MOCK else _run_separator)(job, wav, mode, fmt, out_dir, names)
            _stage(job, 'export', 'Preparando prévias e arquivos…', 'running', 0)
            files = []
            stems = MODES[mode]['stems']
            for i, stem in enumerate(stems):
                _check_cancel(job)
                cand = [f for f in os.listdir(out_dir) if f.lower().startswith(names[stem]) and not f.endswith('.preview.mp3')]
                if not cand:
                    raise RuntimeError(f'A faixa "{STEM_PT[stem]}" não foi gerada pelo separador.' + (' Detalhes: ' + ' | '.join(job.get('log_tail') or [])[-300:] if job.get('log_tail') else ''))
                path = os.path.join(out_dir, f'{base} - {STEM_PT[stem]}{os.path.splitext(cand[0])[1]}')
                os.replace(os.path.join(out_dir, cand[0]), path)
                prev = os.path.join(out_dir, f'{stem.lower()}.preview.mp3')
                _make_preview(path, prev)
                size = os.path.getsize(path)
                files.append({'stem': stem, 'label': STEM_PT[stem], 'icon': STEM_ICON[stem], 'color': STEM_COLOR[stem],
                              'file': os.path.basename(path), 'rel': f"{job['id']}/{os.path.basename(path)}",
                              'preview': f"{job['id']}/{os.path.basename(prev)}" if os.path.exists(prev) else None,
                              'size': size, 'size_label': _fmt_size(size),
                              'drive_path': _copy_to_drive(job, path, os.path.join('Faixas', base))})
                job['percent'] = (i + 1) / len(stems) * 100
            # prévia do original para comparação no mixer
            orig_prev = os.path.join(out_dir, 'original.preview.mp3')
            if _make_preview(wav, orig_prev):
                job['original_preview'] = f"{job['id']}/original.preview.mp3"
            job['files'] = files
            _stage(job, 'export', 'Concluído', 'done', 100)

        def _run_download_job(job):
            src = STATE['sources'][job['source_id']]
            out_dir = os.path.join(OUT_DIR, job['id']); os.makedirs(out_dir, exist_ok=True)
            _stage(job, 'fetch', 'Baixando áudio do YouTube…', 'running', 0)
            path = _download_youtube_audio(job, src, out_dir, job['audio_format'], job['audio_quality'], job['embed_meta'], tag='final')
            _stage(job, 'export', 'Finalizando…', 'running', 50)
            size = os.path.getsize(path)
            job['files'] = [{'stem': 'Audio', 'label': 'Áudio', 'icon': '🎵', 'color': '#38bdf8', 'file': os.path.basename(path),
                             'rel': f"{job['id']}/{os.path.basename(path)}", 'size': size, 'size_label': _fmt_size(size), 'preview': None,
                             'drive_path': _copy_to_drive(job, path, 'Downloads')}]
            _stage(job, 'export', 'Concluído', 'done', 100)

        def _run_job(job_id):
            job = STATE['jobs'][job_id]
            job['started'] = time.time()
            try:
                if job['kind'] == 'separate':
                    _run_separate_job(job)
                else:
                    _run_download_job(job)
                job['status'] = 'done'; job['percent'] = 100
            except yt_dlp.utils.DownloadCancelled:
                job['status'] = 'cancelled'; job['stage_label'] = 'Cancelado'
                for s in job['stages']:
                    if s['status'] == 'running': s['status'] = 'cancelled'
            except Exception as e:
                job['status'] = 'error'; job['error'] = _friendly_error(str(e)); job['stage_label'] = 'Erro'
                for s in job['stages']:
                    if s['status'] == 'running': s['status'] = 'error'
            job['finished'] = time.time()
            job['elapsed'] = round(job['finished'] - job['started'])

        def _worker():
            while True:
                job_id = JOB_QUEUE.get()
                try:
                    job = STATE['jobs'].get(job_id)
                    if job and job.get('cancel'):
                        job['status'] = 'cancelled'; job['stage_label'] = 'Cancelado'; job['finished'] = time.time()
                    elif job:
                        _run_job(job_id)
                except Exception as e:
                    job = STATE['jobs'].get(job_id)
                    if job:
                        job['status'] = 'error'; job['error'] = _friendly_error(str(e)); job['finished'] = time.time()
                finally:
                    JOB_QUEUE.task_done()

        def _ensure_worker():
            w = STATE.get('worker')
            if not w or not w.is_alive():
                w = threading.Thread(target=_worker, daemon=True); w.start()
                STATE['worker'] = w

        def cb_start_job(payload):
            try:
                src = STATE['sources'].get(payload.get('source_id'))
                if not src:
                    return JSON({'ok': False, 'error': 'Fonte não encontrada. Analise o link ou envie o arquivo novamente.'})
                kind = payload.get('kind')
                job_id = uuid.uuid4().hex[:10]
                job = {'id': job_id, 'kind': kind, 'source_id': src['id'], 'title': src['title'], 'thumbnail': src.get('thumbnail'),
                       'status': 'queued', 'cancel': False, 'created': time.time(), 'percent': 0, 'detail': '', 'stage': None,
                       'stage_label': 'Na fila', 'files': [], 'error': None, 'warning': None,
                       'save_drive': bool(payload.get('save_drive')), 'duration': src.get('duration') or 0}
                if kind == 'separate':
                    mode = payload.get('mode'); fmt = payload.get('out_format', 'wav')
                    if mode not in MODES: return JSON({'ok': False, 'error': 'Modo de separação inválido.'})
                    if fmt not in OUT_FORMATS: return JSON({'ok': False, 'error': 'Formato de saída inválido.'})
                    if src.get('too_long'):
                        return JSON({'ok': False, 'error': f'Este áudio tem {src["duration_str"]}. O limite é {MAX_DURATION // 60} minutos.'})
                    job.update({'mode': mode, 'mode_label': MODES[mode]['label'], 'out_format': fmt, 'stems': MODES[mode]['stems'],
                                'stages': [{'key': 'fetch', 'label': 'Obter o áudio', 'status': 'pending', 'percent': 0},
                                           {'key': 'model', 'label': 'Modelo de IA', 'status': 'pending', 'percent': 0},
                                           {'key': 'separate', 'label': 'Separar na GPU', 'status': 'pending', 'percent': 0},
                                           {'key': 'export', 'label': 'Exportar faixas', 'status': 'pending', 'percent': 0}],
                                'label': f"{MODES[mode]['label']} · {fmt.upper()}"})
                    if not MOCK and not _gpu_info()['available']:
                        job['warning'] = 'Sem GPU nesta sessão: a separação vai rodar no processador e pode demorar muitos minutos.'
                elif kind == 'download':
                    if src['kind'] != 'youtube':
                        return JSON({'ok': False, 'error': 'O download de áudio só vale para links do YouTube.'})
                    fmt = payload.get('audio_format', 'mp3')
                    if fmt not in ('mp3', 'm4a', 'opus', 'flac', 'wav', 'best'):
                        return JSON({'ok': False, 'error': 'Formato de áudio inválido.'})
                    job.update({'audio_format': fmt, 'audio_quality': str(payload.get('audio_quality', 'best')),
                                'embed_meta': bool(payload.get('embed_meta', True)),
                                'stages': [{'key': 'fetch', 'label': 'Baixar do YouTube', 'status': 'pending', 'percent': 0},
                                           {'key': 'export', 'label': 'Converter e finalizar', 'status': 'pending', 'percent': 0}],
                                'label': f"Áudio {'original' if fmt == 'best' else fmt.upper()}"})
                else:
                    return JSON({'ok': False, 'error': 'Tipo de tarefa inválido.'})
                STATE['jobs'][job_id] = job
                ahead = sum(1 for j in STATE['jobs'].values() if j['id'] != job_id and j['status'] in ('queued', 'running'))
                JOB_QUEUE.put(job_id); _ensure_worker()
                return JSON({'ok': True, 'job_id': job_id, 'ahead': ahead})
            except Exception as e:
                return JSON({'ok': False, 'error': _friendly_error(str(e))})

        def _job_view(job):
            v = {k: val for k, val in job.items() if k not in ('cancel', 'proc')}
            v['ok'] = True
            if job['status'] == 'running' and job.get('started'):
                v['elapsed'] = round(time.time() - job['started'])
            v['ahead'] = sum(1 for j in STATE['jobs'].values() if j['id'] != job['id'] and j['status'] in ('queued', 'running')
                             and j['created'] < job['created']) if job['status'] == 'queued' else 0
            return v

        def cb_job_status(job_id):
            job = STATE['jobs'].get(job_id)
            if not job:
                return JSON({'ok': False, 'error': 'Tarefa não encontrada.'})
            return JSON(_job_view(job))

        def cb_jobs_list():
            try:
                out = []
                for j in sorted(STATE['jobs'].values(), key=lambda x: x['created']):
                    out.append({'id': j['id'], 'kind': j['kind'], 'status': j['status'], 'label': j.get('label'), 'title': j.get('title'),
                                'thumbnail': j.get('thumbnail'), 'percent': round(j.get('percent') or 0), 'stage_label': j.get('stage_label'),
                                'error': j.get('error'), 'created': j['created'], 'files': len(j.get('files') or [])})
                return JSON({'ok': True, 'jobs': out, 'active': any(j['status'] in ('queued', 'running') for j in STATE['jobs'].values())})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_cancel_job(job_id):
            job = STATE['jobs'].get(job_id)
            if job:
                job['cancel'] = True
                p = job.get('proc')
                if p:
                    try: p.kill()
                    except Exception: pass
            return JSON({'ok': True})

        def cb_zip_job(job_id):
            try:
                job = STATE['jobs'].get(job_id)
                if not job or not job.get('files'): return JSON({'ok': False, 'error': 'Nenhum arquivo para compactar.'})
                src = os.path.join(OUT_DIR, job_id)
                name = _safe_name(job.get('title') or 'faixas', 60)
                tmp = os.path.join(OUT_DIR, f'{job_id}_zipsrc'); shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
                for f in job['files']:
                    shutil.copy2(os.path.join(src, f['file']), os.path.join(tmp, f['file']))
                zp = shutil.make_archive(os.path.join(OUT_DIR, f'{job_id}_{re.sub(r"[^\w.-]+", "_", name)}'), 'zip', tmp)
                shutil.rmtree(tmp, ignore_errors=True)
                drive = _copy_to_drive(job, zp, 'Faixas') if job.get('save_drive') else None
                return JSON({'ok': True, 'rel': os.path.basename(zp), 'size_label': _fmt_size(os.path.getsize(zp)), 'drive_path': drive})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_colab_download(rel):
            """Download pelo mecanismo nativo do Colab (vai direto para a pasta Downloads do navegador)."""
            try:
                path = os.path.normpath(os.path.join(OUT_DIR, rel))
                if not path.startswith(os.path.normpath(OUT_DIR)) or not os.path.exists(path):
                    raise FileNotFoundError('Arquivo não encontrado (a sessão pode ter sido reiniciada).')
                from google.colab import files as colab_files
                colab_files.download(path)
                return JSON({'ok': True})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_cleanup():
            try:
                for j in list(STATE['jobs'].values()):
                    if j['status'] in ('queued', 'running'):
                        return JSON({'ok': False, 'error': 'Há tarefas em andamento. Aguarde ou cancele antes de limpar.'})
                shutil.rmtree(OUT_DIR, ignore_errors=True); os.makedirs(OUT_DIR, exist_ok=True)
                shutil.rmtree(WORK_DIR, ignore_errors=True); os.makedirs(WORK_DIR, exist_ok=True)
                STATE['jobs'].clear(); STATE['sources'].clear()
                return JSON({'ok': True})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        # ---------- pré-download do modelo padrão (em segundo plano) ----------
        def _prefetch_model(model):
            if MOCK or _model_ready(model) or STATE['prefetch'].get(model):
                return
            STATE['prefetch'][model] = 'running'
            def run():
                try:
                    subprocess.run(_sep_cmd() + ['--download_model_only', '--model_filename', model,
                                    '--model_file_dir', MODELS_DIR, '--log_level', 'warning'],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=1800)
                    STATE['prefetch'][model] = 'done' if _model_ready(model) else 'failed'
                except Exception:
                    STATE['prefetch'][model] = 'failed'
            threading.Thread(target=run, daemon=True).start()

        def cb_models_status():
            return JSON({'ok': True, 'models': {k: {'ready': _model_ready(v['model']), 'prefetch': STATE['prefetch'].get(v['model'])} for k, v in MODES.items()}})

        # ---------- servidor de arquivos (prévias do mixer, com suporte a Range) ----------
        class _FileHandler(http.server.SimpleHTTPRequestHandler):
            def __init__(self, *a, **k):
                super().__init__(*a, directory=OUT_DIR, **k)
            def log_message(self, *a): pass
            def list_directory(self, path):
                self.send_error(403, 'Listagem desabilitada'); return None
            def do_GET(self):
                path = self.translate_path(self.path.split('?')[0])
                if os.path.isdir(path) or not os.path.exists(path):
                    return super().do_GET()
                size = os.path.getsize(path)
                rng = self.headers.get('Range')
                start, end = 0, size - 1
                if rng and rng.startswith('bytes='):
                    a, _, b = rng[6:].partition('-')
                    try:
                        start = int(a) if a else max(0, size - int(b))
                        end = int(b) if (a and b) else end
                    except ValueError:
                        start, end = 0, size - 1
                    if start >= size:
                        self.send_response(416); self.send_header('Content-Range', f'bytes */{size}'); self.end_headers(); return
                    end = min(end, size - 1)
                ctype = self.guess_type(path)
                self.send_response(206 if rng else 200)
                self.send_header('Content-Type', ctype)
                self.send_header('Accept-Ranges', 'bytes')
                self.send_header('Content-Length', str(end - start + 1))
                if rng: self.send_header('Content-Range', f'bytes {start}-{end}/{size}')
                self.send_header('Cache-Control', 'no-store')
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                with open(path, 'rb') as f:
                    f.seek(start); remaining = end - start + 1
                    while remaining > 0:
                        chunk = f.read(min(65536, remaining))
                        if not chunk: break
                        try: self.wfile.write(chunk)
                        except (BrokenPipeError, ConnectionResetError): break
                        remaining -= len(chunk)

        def _start_file_server():
            if STATE.get('server'): return
            class _Srv(socketserver.ThreadingTCPServer):
                allow_reuse_address = True; daemon_threads = True
            try:
                srv = _Srv(('0.0.0.0', FILE_PORT), _FileHandler)
            except OSError:
                return
            STATE['server'] = srv
            threading.Thread(target=srv.serve_forever, daemon=True).start()

        _start_file_server()
        _prefetch_model(MODES['2stem']['model'])

        # ---------- registro das funções para a interface ----------
        def _register(name, fn):
            try:
                from google.colab import output as _out
                _out.register_callback(f'stemlab.{name}', fn)
            except Exception:
                pass

        for _n, _f in {
            'bootstrap': cb_bootstrap, 'save_cookies': cb_save_cookies, 'clear_cookies': cb_clear_cookies, 'test_cookies': cb_test_cookies,
            'analyze': cb_analyze, 'upload_start': cb_upload_start, 'upload_chunk': cb_upload_chunk, 'upload_finish': cb_upload_finish,
            'start_job': cb_start_job, 'job_status': cb_job_status, 'jobs_list': cb_jobs_list, 'cancel_job': cb_cancel_job,
            'zip_job': cb_zip_job, 'colab_download': cb_colab_download, 'cleanup': cb_cleanup, 'models_status': cb_models_status,
        }.items():
            _register(_n, _f)

        APP_HTML = r"""<link rel="preconnect" href="https://fonts.googleapis.com">
        <link href="https://fonts.googleapis.com/css2?family=Sora:wght@500;600;700;800&family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
        <style>
          #stemlab, #stemlab * { box-sizing: border-box; }
          #stemlab {
            --bg: #0b0d14; --bg-2: #10131c; --panel: rgba(22, 25, 37, .78); --panel-solid: #161925; --panel-2: #1c2030; --line: rgba(255,255,255,.08); --line-2: rgba(255,255,255,.16);
            --text: #f1f3fa; --muted: #a0a8bf; --muted-2: #6d7590;
            --a1: #8b5cf6; --a2: #22d3ee; --a3: #f472b6;
            --accent: var(--a1); --accent-soft: rgba(139,92,246,.16);
            --ok: #34d399; --ok-soft: rgba(52,211,153,.14); --warn: #fbbf24; --warn-soft: rgba(251,191,36,.14);
            --err: #f87171; --err-soft: rgba(248,113,113,.14); --info: #60a5fa; --info-soft: rgba(96,165,250,.14);
            --radius: 18px; --radius-sm: 12px; --shadow: 0 18px 50px rgba(0,0,0,.45); --glow: 0 0 0 1px rgba(255,255,255,.04), 0 12px 40px rgba(139,92,246,.18);
            font-family: 'Inter', system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif;
            color: var(--text); background: var(--bg); border-radius: 26px; overflow: hidden;
            max-width: 1120px; margin: 0 auto; min-height: 600px; position: relative; line-height: 1.5;
            -webkit-font-smoothing: antialiased; isolation: isolate;
          }
          #stemlab[data-theme="light"] {
            --bg: #f3f4fb; --bg-2: #eceef8; --panel: rgba(255,255,255,.82); --panel-solid: #ffffff; --panel-2: #f1f2fa; --line: rgba(20,24,40,.08); --line-2: rgba(20,24,40,.18);
            --text: #12152a; --muted: #5a6280; --muted-2: #8b93ad; --shadow: 0 18px 50px rgba(20,24,60,.10); --glow: 0 0 0 1px rgba(20,24,60,.04), 0 12px 40px rgba(139,92,246,.12);
          }
          #stemlab h1, #stemlab h2, #stemlab .display { font-family: 'Sora', 'Inter', system-ui, sans-serif; letter-spacing: -.02em; }

          /* ---- fundo animado ---- */
          #stemlab .aurora { position:absolute; inset:0; pointer-events:none; overflow:hidden; z-index:-1; }
          #stemlab .aurora i { position:absolute; border-radius:50%; filter: blur(60px); opacity:.55; animation: slFloat 16s ease-in-out infinite; }
          #stemlab .aurora i:nth-child(1) { width:560px; height:560px; left:-180px; top:-260px; background: radial-gradient(circle, rgba(139,92,246,.55), transparent 65%); }
          #stemlab .aurora i:nth-child(2) { width:520px; height:520px; right:-200px; top:-120px; background: radial-gradient(circle, rgba(34,211,238,.38), transparent 65%); animation-delay:-5s; }
          #stemlab .aurora i:nth-child(3) { width:460px; height:460px; left:35%; bottom:-280px; background: radial-gradient(circle, rgba(244,114,182,.30), transparent 65%); animation-delay:-9s; }
          #stemlab[data-theme="light"] .aurora i { opacity:.35; }
          @keyframes slFloat { 0%,100% { transform: translate(0,0) scale(1);} 50% { transform: translate(40px,30px) scale(1.08);} }
          #stemlab .grain { position:absolute; inset:0; pointer-events:none; z-index:-1; opacity:.06; background-image: radial-gradient(rgba(255,255,255,.6) .6px, transparent .6px); background-size: 3px 3px; }

          /* ---- topo ---- */
          #stemlab .topbar { display:flex; align-items:center; gap:12px; flex-wrap:wrap; padding:16px 22px; border-bottom:1px solid var(--line); background: var(--panel); backdrop-filter: blur(14px); }
          #stemlab .brand { display:flex; align-items:center; gap:12px; margin-right:auto; }
          #stemlab .logo { width:44px; height:44px; border-radius:13px; display:grid; place-items:center; background: linear-gradient(135deg, var(--a1), var(--a2)); box-shadow: 0 8px 22px rgba(139,92,246,.4); position:relative; overflow:hidden; }
          #stemlab .logo::after { content:''; position:absolute; inset:0; background: linear-gradient(120deg, transparent 30%, rgba(255,255,255,.35) 50%, transparent 70%); transform: translateX(-100%); animation: slShine 4s ease-in-out infinite; }
          @keyframes slShine { 0%,60% { transform: translateX(-100%);} 100% { transform: translateX(100%);} }
          #stemlab .logo svg { width:24px; height:24px; }
          #stemlab .brand h1 { font-size:18px; margin:0; font-weight:800; }
          #stemlab .brand h1 span { background: linear-gradient(90deg, var(--a1), var(--a2)); -webkit-background-clip:text; background-clip:text; color:transparent; }
          #stemlab .brand p { margin:0; font-size:12px; color:var(--muted); }
          #stemlab .status { display:flex; gap:8px; flex-wrap:wrap; align-items:center; }
          #stemlab .chip { display:inline-flex; align-items:center; gap:7px; padding:6px 11px; border-radius:999px; font-size:12px; font-weight:600; border:1px solid var(--line); background:var(--panel-2); color:var(--muted); white-space:nowrap; transition: all .2s; }
          #stemlab .chip .dot { width:8px; height:8px; border-radius:50%; background:var(--muted-2); flex:none; }
          #stemlab .chip.ok { color:var(--ok); border-color: rgba(52,211,153,.35); background:var(--ok-soft);} #stemlab .chip.ok .dot{background:var(--ok); box-shadow:0 0 8px var(--ok)}
          #stemlab .chip.warn { color:var(--warn); border-color: rgba(251,191,36,.35); background:var(--warn-soft);} #stemlab .chip.warn .dot{background:var(--warn)}
          #stemlab .chip.err { color:var(--err); background:var(--err-soft);} #stemlab .chip.err .dot{background:var(--err)}
          #stemlab .chip.info { color:var(--info); background:var(--info-soft);} #stemlab .chip.info .dot{background:var(--info)}
          #stemlab .chip.pulse .dot { animation: slPulse 1.2s ease-in-out infinite; }
          @keyframes slPulse { 0%,100% { opacity:1; transform:scale(1);} 50% { opacity:.4; transform:scale(.7);} }
          #stemlab .chip.clickable { cursor:pointer; } #stemlab .chip.clickable:hover { border-color: var(--line-2); transform: translateY(-1px); }
          #stemlab .user { display:flex; align-items:center; gap:9px; padding:5px 12px 5px 5px; border-radius:999px; border:1px solid var(--line); background:var(--panel-2); font-size:12.5px; font-weight:600; max-width:100%; }
          #stemlab .avatar { width:28px; height:28px; border-radius:50%; display:grid; place-items:center; font-size:12px; font-weight:800; color:#fff; background: linear-gradient(135deg, var(--a2), var(--a1)); flex:none; }
          #stemlab .user span { overflow:hidden; text-overflow:ellipsis; white-space:nowrap; max-width:200px; }
          #stemlab .icon-btn { width:36px; height:36px; border-radius:10px; border:1px solid var(--line); background:var(--panel-2); color:var(--text); cursor:pointer; display:grid; place-items:center; font-size:16px; transition: all .15s; }
          #stemlab .icon-btn:hover { border-color:var(--line-2); transform: rotate(15deg); }

          /* ---- etapas ---- */
          #stemlab .stepper { display:flex; gap:6px; padding:16px 22px 0; overflow-x:auto; scrollbar-width:none; }
          #stemlab .stepper::-webkit-scrollbar{display:none}
          #stemlab .step { flex:1; min-width:118px; display:flex; align-items:center; gap:10px; padding:10px 12px; border-radius:12px; border:1px solid transparent; color:var(--muted-2); font-size:12.5px; font-weight:600; cursor:default; transition: all .25s; position:relative; }
          #stemlab .step .n { width:26px; height:26px; border-radius:50%; display:grid; place-items:center; font-size:12px; font-weight:800; background:var(--panel-2); border:1px solid var(--line); flex:none; transition: all .25s; }
          #stemlab .step.done { color:var(--muted); cursor:pointer; } #stemlab .step.done .n { background:var(--ok-soft); border-color:transparent; color:var(--ok); }
          #stemlab .step.active { color:var(--text); background:var(--panel); border-color:var(--line); box-shadow:var(--shadow); backdrop-filter: blur(10px); }
          #stemlab .step.active .n { background: linear-gradient(135deg, var(--a1), var(--a2)); border-color:transparent; color:#fff; box-shadow: 0 0 14px rgba(139,92,246,.6); }
          #stemlab .step.done:hover { background:var(--panel); }

          /* ---- fila ---- */
          #stemlab .queuebar { margin:16px 22px 0; padding:12px 14px; border-radius:14px; border:1px solid var(--line); background: var(--panel); box-shadow: var(--shadow); backdrop-filter: blur(10px); }
          #stemlab .queuebar .qhead { display:flex; align-items:center; gap:10px; flex-wrap:wrap; font-weight:800; font-size:13.5px; margin-bottom:8px; }
          #stemlab .qlist { display:grid; gap:6px; }
          #stemlab .qjob { display:grid; grid-template-columns: 26px 1fr 130px auto; gap:10px; align-items:center; padding:8px 10px; border-radius:10px; background:var(--panel-2); border:1px solid var(--line); font-size:12.5px; }
          #stemlab .qjob > div { min-width:0; }
          #stemlab .qjob.viewing { border-color:var(--accent); }
          #stemlab .qjob .ql { font-weight:700; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; }
          #stemlab .qjob .qs { color:var(--muted); font-size:11.5px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; margin-top:1px; }
          #stemlab .qjob .bar { height:6px; }
          #stemlab .st { width:24px; height:24px; border-radius:50%; display:grid; place-items:center; font-size:12px; font-weight:900; background:var(--panel-solid); border:1px solid var(--line); color:var(--muted-2); }
          #stemlab .st.done { background:var(--ok-soft); color:var(--ok); border-color:transparent; }
          #stemlab .st.warn { background:var(--warn-soft); color:var(--warn); border-color:transparent; }
          #stemlab .st.err { background:var(--err-soft); color:var(--err); border-color:transparent; }
          #stemlab .st.run { border-color:transparent; background:var(--accent-soft); }
          #stemlab .st.run::after { content:''; width:11px; height:11px; border:2px solid var(--accent-soft); border-top-color:var(--accent); border-radius:50%; animation: slSpin .8s linear infinite; }
          @media (max-width: 600px) { #stemlab .queuebar { margin:12px 14px 0; } #stemlab .qjob { grid-template-columns: 24px 1fr auto; } #stemlab .qjob .bar { display:none; } }

          /* ---- conteúdo ---- */
          #stemlab main { padding:20px 22px 26px; }
          #stemlab section[data-step] { display:none; animation: slFade .3s ease; }
          #stemlab section[data-step].on { display:block; }
          @keyframes slFade { from { opacity:0; transform: translateY(8px);} to { opacity:1; transform:none;} }
          #stemlab .card { background:var(--panel); border:1px solid var(--line); border-radius:var(--radius); padding:22px; box-shadow:var(--shadow); backdrop-filter: blur(12px); }
          #stemlab .card + .card, #stemlab .card + details.card { margin-top:16px; }
          #stemlab h2 { font-size:21px; margin:0 0 6px; font-weight:800; }
          #stemlab h3 { font-size:14px; margin:0 0 10px; font-weight:700; color:var(--text); }
          #stemlab .sub { margin:0 0 16px; color:var(--muted); font-size:13.5px; }
          #stemlab .sub b, #stemlab .note b, #stemlab .steps-help b { color:var(--text); font-weight:600; }
          #stemlab .hero { text-align:center; padding:30px 20px 26px; position:relative; overflow:hidden; }
          #stemlab .hero h2 { font-size:30px; line-height:1.15; margin-bottom:10px; }
          #stemlab .hero h2 em { font-style:normal; background: linear-gradient(90deg, var(--a1), var(--a2) 60%, var(--a3)); -webkit-background-clip:text; background-clip:text; color:transparent; }
          #stemlab .hero .sub { font-size:14.5px; max-width:600px; margin:0 auto 20px; }
          #stemlab .eq { display:flex; gap:4px; align-items:flex-end; justify-content:center; height:54px; margin:0 auto 18px; }
          #stemlab .eq i { width:6px; border-radius:4px; background: linear-gradient(180deg, var(--a2), var(--a1)); animation: slEq 1.4s ease-in-out infinite; opacity:.85; }
          @keyframes slEq { 0%,100% { height:14%; } 50% { height:100%; } }
          #stemlab .tabs { display:inline-flex; gap:4px; padding:4px; border-radius:14px; background:var(--panel-2); border:1px solid var(--line); margin-bottom:18px; }
          #stemlab .tab { padding:9px 16px; border-radius:10px; border:0; background:transparent; color:var(--muted); font:inherit; font-weight:700; font-size:13px; cursor:pointer; transition: all .2s; display:inline-flex; gap:8px; align-items:center; }
          #stemlab .tab.sel { background:var(--panel-solid); color:var(--text); box-shadow: 0 4px 14px rgba(0,0,0,.25); }
          #stemlab .src-pane { display:none; } #stemlab .src-pane.on { display:block; animation: slFade .25s ease; }

          #stemlab .input-row { display:flex; gap:10px; align-items:stretch; max-width:760px; margin:0 auto; }
          #stemlab .field { position:relative; flex:1; display:flex; align-items:center; }
          #stemlab .field svg { position:absolute; left:14px; width:18px; height:18px; fill:var(--muted-2); pointer-events:none; }
          #stemlab input[type=text], #stemlab textarea { width:100%; background:var(--panel-2); border:1px solid var(--line); color:var(--text); border-radius:12px; padding:14px 14px 14px 42px; font:inherit; font-size:14.5px; outline:none; transition: border .15s, box-shadow .15s; }
          #stemlab textarea { padding:12px 14px; min-height:110px; resize:vertical; font-family: ui-monospace, Menlo, Consolas, monospace; font-size:12px; }
          #stemlab input[type=text].plain { padding-left:14px; }
          #stemlab input[type=text]:focus, #stemlab textarea:focus { border-color:var(--accent); box-shadow: 0 0 0 3px var(--accent-soft); }
          #stemlab input::placeholder { color:var(--muted-2); }
          #stemlab .drop { max-width:760px; margin:0 auto; padding:34px 20px; border-radius:16px; border:2px dashed var(--line-2); background:var(--panel-2); cursor:pointer; transition: all .2s; position:relative; }
          #stemlab .drop:hover, #stemlab .drop.over { border-color:var(--accent); background: color-mix(in srgb, var(--accent-soft) 50%, var(--panel-2)); transform: scale(1.005); }
          #stemlab .drop .big { font-size:34px; margin-bottom:8px; }
          #stemlab .drop b { display:block; font-size:15px; }
          #stemlab .drop span { color:var(--muted); font-size:12.5px; }
          #stemlab .drop input { position:absolute; inset:0; opacity:0; cursor:pointer; }
          #stemlab .upbar { max-width:760px; margin:12px auto 0; display:none; }
          #stemlab .upbar.on { display:block; }

          #stemlab .btn { display:inline-flex; align-items:center; justify-content:center; gap:8px; padding:12px 18px; border-radius:12px; border:1px solid var(--line); background:var(--panel-2); color:var(--text); font:inherit; font-size:14px; font-weight:700; cursor:pointer; transition: transform .1s, background .15s, border-color .15s, opacity .15s, box-shadow .2s; white-space:nowrap; text-decoration:none; }
          #stemlab .btn:hover { border-color:var(--line-2); transform: translateY(-1px); } #stemlab .btn:active { transform: translateY(1px) scale(.99); }
          #stemlab .btn.primary { background: linear-gradient(135deg, var(--a1), #6366f1 55%, var(--a2)); background-size: 200% 200%; border-color:transparent; color:#fff; box-shadow: 0 10px 26px rgba(139,92,246,.35); animation: slGrad 6s ease infinite; }
          @keyframes slGrad { 0%,100% { background-position: 0% 50%; } 50% { background-position: 100% 50%; } }
          #stemlab .btn.primary:hover { filter: brightness(1.08); box-shadow: 0 14px 32px rgba(139,92,246,.45); }
          #stemlab .btn.ghost { background:transparent; }
          #stemlab .btn.danger { color:var(--err); border-color: rgba(248,113,113,.4); background:var(--err-soft); }
          #stemlab .btn.sm { padding:8px 12px; font-size:12.5px; border-radius:10px; }
          #stemlab .btn.lg { padding:15px 28px; font-size:15px; border-radius:14px; }
          #stemlab .btn:disabled { opacity:.45; cursor:not-allowed; transform:none; filter:none; animation:none; }
          #stemlab .btn .spin, #stemlab .spin { width:16px; height:16px; border:2px solid rgba(255,255,255,.35); border-top-color:#fff; border-radius:50%; animation: slSpin .7s linear infinite; display:inline-block; }
          @keyframes slSpin { to { transform: rotate(360deg);} }
          #stemlab .hint { font-size:12.5px; color:var(--muted-2); margin-top:12px; }
          #stemlab .hint b { color:var(--muted); font-weight:700; }
          #stemlab .actions { display:flex; gap:10px; justify-content:space-between; align-items:center; flex-wrap:wrap; margin-top:20px; }
          #stemlab .actions .right { display:flex; gap:10px; margin-left:auto; flex-wrap:wrap; }
          #stemlab .row { display:flex; gap:10px; flex-wrap:wrap; align-items:center; }
          #stemlab .small { font-size:12.5px; } #stemlab .muted { color:var(--muted); }
          #stemlab .badge { display:inline-flex; align-items:center; gap:5px; padding:3px 9px; border-radius:999px; font-size:11.5px; font-weight:700; background:var(--panel-2); border:1px solid var(--line); color:var(--muted); }
          #stemlab .badge.ok { color:var(--ok); background:var(--ok-soft); border-color:transparent; }
          #stemlab .badge.warn { color:var(--warn); background:var(--warn-soft); border-color:transparent; }
          #stemlab .badge.acc { color:#fff; background: linear-gradient(135deg, var(--a1), var(--a2)); border-color:transparent; }

          /* ---- colapsável ---- */
          #stemlab details.card { padding:0; }
          #stemlab details.card > summary { list-style:none; cursor:pointer; padding:16px 20px; display:flex; align-items:center; gap:12px; font-weight:700; font-size:14px; }
          #stemlab details.card > summary::-webkit-details-marker { display:none; }
          #stemlab details.card > summary .chev { margin-left:auto; transition: transform .2s; color:var(--muted); }
          #stemlab details.card[open] > summary .chev { transform: rotate(180deg); }
          #stemlab details.card > .body { padding:0 20px 20px; border-top:1px solid var(--line); padding-top:16px; }
          #stemlab .steps-help { counter-reset: s; margin:0; padding:0; list-style:none; display:grid; gap:10px; }
          #stemlab .steps-help li { display:flex; gap:12px; font-size:13px; color:var(--muted); align-items:flex-start; }
          #stemlab .steps-help li::before { counter-increment:s; content: counter(s); width:22px; height:22px; border-radius:50%; background:var(--accent-soft); color:var(--a1); font-weight:800; font-size:11.5px; display:grid; place-items:center; flex:none; margin-top:1px; }
          #stemlab .steps-help a, #stemlab .note a, #stemlab .sub a { color:var(--info); text-decoration:none; font-weight:600; }
          #stemlab .note { font-size:12.5px; color:var(--muted); padding:12px 14px; border-radius:12px; background:var(--panel-2); border:1px dashed var(--line-2); margin-top:14px; }
          #stemlab .note.warn { border-color: rgba(251,191,36,.5); background:var(--warn-soft); color: var(--text); border-style:solid; }
          #stemlab .note.info { border-color: rgba(96,165,250,.5); background:var(--info-soft); color: var(--text); border-style:solid; }
          #stemlab .note.err { border-color: rgba(248,113,113,.5); background:var(--err-soft); color: var(--text); border-style:solid; }

          /* ---- prévia ---- */
          #stemlab .preview { display:grid; grid-template-columns: 300px 1fr; gap:20px; align-items:start; }
          #stemlab .thumb { position:relative; border-radius:14px; overflow:hidden; background:#000; aspect-ratio:16/9; cursor:pointer; box-shadow: var(--shadow); }
          #stemlab .thumb img { width:100%; height:100%; object-fit:cover; display:block; transition: transform .4s; }
          #stemlab .thumb:hover img { transform: scale(1.04); }
          #stemlab .thumb .play { position:absolute; inset:0; display:grid; place-items:center; background: rgba(0,0,0,.25); }
          #stemlab .thumb .play i { width:58px; height:58px; border-radius:50%; background: rgba(255,255,255,.92); display:grid; place-items:center; color:#111; font-size:22px; box-shadow: 0 8px 24px rgba(0,0,0,.4); transition: transform .2s; }
          #stemlab .thumb:hover .play i { transform: scale(1.08); }
          #stemlab .thumb iframe { position:absolute; inset:0; width:100%; height:100%; border:0; }
          #stemlab .thumb .dur { position:absolute; right:8px; bottom:8px; background: rgba(0,0,0,.75); color:#fff; font-size:11.5px; font-weight:700; padding:2px 7px; border-radius:6px; }
          #stemlab .filecard { border-radius:14px; background: linear-gradient(135deg, rgba(139,92,246,.25), rgba(34,211,238,.18)); aspect-ratio:16/9; display:grid; place-items:center; font-size:52px; position:relative; }
          #stemlab .meta h2 { font-size:19px; line-height:1.3; }
          #stemlab .meta .chan { color:var(--muted); font-size:13px; margin-bottom:12px; }
          #stemlab .stats { display:flex; gap:8px; flex-wrap:wrap; margin-bottom:14px; }
          @media (max-width: 720px) { #stemlab .preview { grid-template-columns: 1fr; } }

          /* ---- cartões de escolha ---- */
          #stemlab .choices { display:grid; grid-template-columns: repeat(auto-fit, minmax(230px, 1fr)); gap:12px; }
          #stemlab .choice { position:relative; text-align:left; padding:18px; border-radius:16px; border:1px solid var(--line); background:var(--panel-2); cursor:pointer; transition: all .2s; color:var(--text); font:inherit; overflow:hidden; }
          #stemlab .choice::before { content:''; position:absolute; inset:0; background: linear-gradient(135deg, rgba(139,92,246,.18), rgba(34,211,238,.08)); opacity:0; transition: opacity .2s; }
          #stemlab .choice:hover { border-color:var(--line-2); transform: translateY(-2px); box-shadow: var(--shadow); }
          #stemlab .choice:hover::before { opacity:.6; }
          #stemlab .choice.sel { border-color:var(--a1); box-shadow: 0 0 0 3px var(--accent-soft), var(--glow); }
          #stemlab .choice.sel::before { opacity:1; }
          #stemlab .choice > * { position:relative; }
          #stemlab .choice .ic { font-size:26px; margin-bottom:8px; display:block; }
          #stemlab .choice .t { font-weight:800; font-size:15px; display:block; font-family:'Sora',sans-serif; }
          #stemlab .choice .d { display:block; font-size:12.5px; color:var(--muted); margin-top:4px; line-height:1.45; }
          #stemlab .choice .tag { position:absolute; top:12px; right:12px; }
          #stemlab .choice.big { padding:24px; } #stemlab .choice.big .ic { font-size:34px; } #stemlab .choice.big .t { font-size:18px; }
          #stemlab .choice.sel .check { position:absolute; top:12px; right:12px; width:22px; height:22px; border-radius:50%; background: linear-gradient(135deg, var(--a1), var(--a2)); color:#fff; display:grid; place-items:center; font-size:12px; font-weight:900; }
          #stemlab .choice:not(.sel) .check { display:none; }
          #stemlab .pills { display:flex; gap:6px; flex-wrap:wrap; margin-top:10px; }
          #stemlab .pill { font-size:11.5px; font-weight:700; padding:3px 9px; border-radius:999px; color:#0b0d14; }
          #stemlab .chips { display:flex; gap:8px; flex-wrap:wrap; }
          #stemlab .chipbtn { padding:9px 14px; border-radius:999px; border:1px solid var(--line); background:var(--panel-2); color:var(--muted); font:inherit; font-size:13px; font-weight:700; cursor:pointer; transition: all .15s; }
          #stemlab .chipbtn:hover { border-color:var(--line-2); color:var(--text); }
          #stemlab .chipbtn.sel { border-color:transparent; color:#fff; background: linear-gradient(135deg, var(--a1), var(--a2)); box-shadow: 0 6px 16px rgba(139,92,246,.35); }
          #stemlab .chipbtn small { display:block; font-weight:500; font-size:11px; opacity:.85; }
          #stemlab .toggle { display:flex; align-items:center; justify-content:space-between; gap:14px; padding:14px 0; border-top:1px solid var(--line); cursor:pointer; }
          #stemlab .toggle:first-of-type { border-top:0; padding-top:0; }
          #stemlab .toggle .tt { font-weight:700; font-size:14px; } #stemlab .toggle .td { font-size:12.5px; color:var(--muted); margin-top:2px; }
          #stemlab .sw { width:46px; height:26px; border-radius:999px; background:var(--line-2); position:relative; flex:none; transition: background .2s; }
          #stemlab .sw::after { content:''; position:absolute; top:3px; left:3px; width:20px; height:20px; border-radius:50%; background:#fff; transition: transform .2s; box-shadow: 0 2px 6px rgba(0,0,0,.3); }
          #stemlab .sw.on { background: linear-gradient(135deg, var(--a1), var(--a2)); } #stemlab .sw.on::after { transform: translateX(20px); }
          #stemlab .toggle.off { cursor:default; opacity:.7; }
          #stemlab .summary { display:grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap:10px; }
          #stemlab .summary div { padding:12px 14px; border-radius:12px; background:var(--panel-2); border:1px solid var(--line); }
          #stemlab .summary .k { font-size:11px; text-transform:uppercase; letter-spacing:.06em; color:var(--muted-2); font-weight:700; }
          #stemlab .summary .v { font-weight:700; font-size:14px; margin-top:3px; word-break:break-word; }

          /* ---- processamento ---- */
          #stemlab .proc { display:grid; grid-template-columns: 200px 1fr; gap:26px; align-items:center; }
          #stemlab .ring { width:200px; height:200px; position:relative; display:grid; place-items:center; margin:0 auto; }
          #stemlab .ring svg { position:absolute; inset:0; transform: rotate(-90deg); }
          #stemlab .ring circle { fill:none; stroke-width:10; stroke-linecap:round; }
          #stemlab .ring .track { stroke: var(--line-2); }
          #stemlab .ring .prog { stroke: url(#slGrad); transition: stroke-dashoffset .6s ease; filter: drop-shadow(0 0 8px rgba(139,92,246,.6)); }
          #stemlab .ring .pct { font-family:'Sora',sans-serif; font-size:40px; font-weight:800; line-height:1; }
          #stemlab .ring .pct small { display:block; font-size:12px; color:var(--muted); font-weight:600; margin-top:6px; font-family:'Inter',sans-serif; }
          #stemlab .ring.busy .prog { animation: slRingBusy 1.6s linear infinite; }
          @keyframes slRingBusy { from { stroke-dashoffset: 530; transform: rotate(-90deg);} to { stroke-dashoffset: 530; transform: rotate(270deg);} }
          #stemlab .timeline { display:grid; gap:10px; }
          #stemlab .tl { display:grid; grid-template-columns: 30px 1fr auto; gap:12px; align-items:center; padding:12px 14px; border-radius:12px; border:1px solid var(--line); background:var(--panel-2); transition: all .25s; }
          #stemlab .tl.running { border-color: rgba(139,92,246,.5); background: color-mix(in srgb, var(--accent-soft) 40%, var(--panel-2)); box-shadow: 0 0 0 3px var(--accent-soft); }
          #stemlab .tl.done { opacity:.75; }
          #stemlab .tl .lab { font-weight:700; font-size:13.5px; } #stemlab .tl .det { font-size:12px; color:var(--muted); margin-top:1px; min-height:14px; }
          #stemlab .tl .tm { font-size:12px; color:var(--muted-2); font-variant-numeric: tabular-nums; }
          #stemlab .bar { height:8px; border-radius:999px; background:var(--line); overflow:hidden; position:relative; }
          #stemlab .bar i { display:block; height:100%; width:0; border-radius:999px; background: linear-gradient(90deg, var(--a1), var(--a2)); transition: width .4s ease; }
          #stemlab .bar.ok i { background: linear-gradient(90deg, var(--ok), #6ee7b7); }
          #stemlab .bar.striped i { background-image: linear-gradient(90deg, var(--a1), var(--a2)), repeating-linear-gradient(45deg, rgba(255,255,255,.18) 0 8px, transparent 8px 16px); background-blend-mode: overlay; animation: slStripe 1s linear infinite; }
          @keyframes slStripe { to { background-position: 0 0, 32px 0; } }
          @media (max-width: 720px) { #stemlab .proc { grid-template-columns: 1fr; } }

          /* ---- mixer ---- */
          #stemlab .transport { display:grid; grid-template-columns: auto 1fr auto; gap:14px; align-items:center; padding:14px 16px; border-radius:14px; background:var(--panel-2); border:1px solid var(--line); }
          #stemlab .playbtn { width:52px; height:52px; border-radius:50%; border:0; cursor:pointer; color:#fff; font-size:20px; display:grid; place-items:center; background: linear-gradient(135deg, var(--a1), var(--a2)); box-shadow: 0 10px 24px rgba(139,92,246,.4); transition: transform .15s; }
          #stemlab .playbtn:hover { transform: scale(1.06); } #stemlab .playbtn:disabled { opacity:.5; }
          #stemlab .seek { -webkit-appearance:none; appearance:none; width:100%; height:8px; border-radius:999px; background: linear-gradient(90deg, var(--a1) var(--p, 0%), var(--line) var(--p, 0%)); outline:none; cursor:pointer; }
          #stemlab .seek::-webkit-slider-thumb { -webkit-appearance:none; width:18px; height:18px; border-radius:50%; background:#fff; box-shadow: 0 2px 8px rgba(0,0,0,.4); border:3px solid var(--a1); }
          #stemlab .seek::-moz-range-thumb { width:18px; height:18px; border-radius:50%; background:#fff; border:3px solid var(--a1); }
          #stemlab .time { font-variant-numeric: tabular-nums; font-size:12.5px; color:var(--muted); font-weight:600; white-space:nowrap; }
          #stemlab .stems { display:grid; gap:10px; margin-top:14px; }
          #stemlab .stem { display:grid; grid-template-columns: 44px 1fr auto; gap:12px; align-items:center; padding:12px 14px; border-radius:14px; background:var(--panel-2); border:1px solid var(--line); border-left:4px solid var(--c, var(--a1)); transition: all .2s; }
          #stemlab .stem.muted { opacity:.55; }
          #stemlab .stem .ico { width:44px; height:44px; border-radius:12px; display:grid; place-items:center; font-size:22px; background: color-mix(in srgb, var(--c) 22%, transparent); }
          #stemlab .stem .nm { font-weight:800; font-size:14.5px; font-family:'Sora',sans-serif; display:flex; align-items:center; gap:8px; flex-wrap:wrap; }
          #stemlab .stem .fi { font-size:12px; color:var(--muted); margin-top:2px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; }
          #stemlab .stem .ctl { display:flex; align-items:center; gap:8px; flex-wrap:wrap; justify-content:flex-end; }
          #stemlab .mbtn { width:34px; height:34px; border-radius:9px; border:1px solid var(--line); background:var(--panel-solid); color:var(--muted); font-weight:800; font-size:12px; cursor:pointer; transition: all .15s; }
          #stemlab .mbtn.on.m { background:var(--err-soft); color:var(--err); border-color:transparent; }
          #stemlab .mbtn.on.s { background:var(--warn-soft); color:var(--warn); border-color:transparent; }
          #stemlab .vol { -webkit-appearance:none; appearance:none; width:110px; height:6px; border-radius:999px; background: linear-gradient(90deg, var(--c) var(--p, 100%), var(--line) var(--p, 100%)); outline:none; cursor:pointer; }
          #stemlab .vol::-webkit-slider-thumb { -webkit-appearance:none; width:14px; height:14px; border-radius:50%; background:#fff; border:2px solid var(--c); }
          #stemlab .vol::-moz-range-thumb { width:14px; height:14px; border-radius:50%; background:#fff; border:2px solid var(--c); }
          #stemlab .meter { display:flex; gap:2px; align-items:flex-end; height:18px; width:36px; }
          #stemlab .meter i { flex:1; background: var(--c); border-radius:2px; height:20%; opacity:.5; transition: height .1s; }
          #stemlab .stem.playing .meter i { animation: slMeter .9s ease-in-out infinite; opacity:.9; }
          #stemlab .stem.playing .meter i:nth-child(2) { animation-delay:-.3s; } #stemlab .stem.playing .meter i:nth-child(3) { animation-delay:-.6s; } #stemlab .stem.playing .meter i:nth-child(4) { animation-delay:-.15s; }
          @keyframes slMeter { 0%,100% { height:25%; } 50% { height:95%; } }
          @media (max-width: 640px) { #stemlab .stem { grid-template-columns: 40px 1fr; } #stemlab .stem .ctl { grid-column: 1 / -1; justify-content:flex-start; } #stemlab .vol { width:100%; flex:1; } }
          #stemlab .result-head { display:flex; align-items:center; gap:16px; flex-wrap:wrap; }
          #stemlab .result-head .big { width:62px; height:62px; border-radius:18px; display:grid; place-items:center; font-size:30px; background: linear-gradient(135deg, var(--a1), var(--a2)); box-shadow: 0 10px 26px rgba(139,92,246,.4); animation: slPop .5s cubic-bezier(.2,1.4,.4,1); }
          #stemlab .result-head .big.err { background: linear-gradient(135deg, var(--err), #fb923c); } #stemlab .result-head .big.warn { background: linear-gradient(135deg, var(--warn), #fb923c); }
          @keyframes slPop { from { transform: scale(.4); opacity:0; } to { transform: scale(1); opacity:1; } }
          #stemlab .confetti { position:absolute; width:8px; height:8px; border-radius:2px; top:-10px; animation: slConf 1.8s ease-out forwards; pointer-events:none; }
          @keyframes slConf { to { transform: translateY(420px) rotate(540deg); opacity:0; } }

          /* ---- rodapé, toasts, modal ---- */
          #stemlab footer { display:flex; gap:12px; flex-wrap:wrap; justify-content:space-between; align-items:center; padding:14px 22px; border-top:1px solid var(--line); font-size:12px; color:var(--muted-2); }
          #stemlab footer .signature { font-weight:700; color:var(--muted); }
          #stemlab footer .signature b { background: linear-gradient(90deg, var(--a1), var(--a2)); -webkit-background-clip:text; background-clip:text; color:transparent; }
          #stemlab .toasts { position:fixed; right:18px; bottom:18px; display:grid; gap:8px; z-index:1000; max-width:min(420px, calc(100vw - 36px)); }
          #stemlab .toast { display:flex; gap:10px; align-items:flex-start; padding:12px 14px; border-radius:12px; background:var(--panel-solid); border:1px solid var(--line); color:var(--text); box-shadow: var(--shadow); font-size:13px; animation: slToast .25s ease; }
          @keyframes slToast { from { transform: translateY(10px); opacity:0; } to { transform:none; opacity:1; } }
          #stemlab .toast.ok { border-left:4px solid var(--ok);} #stemlab .toast.err { border-left:4px solid var(--err);} #stemlab .toast.warn { border-left:4px solid var(--warn);} #stemlab .toast.info { border-left:4px solid var(--info);}
          #stemlab .overlay { position:fixed; inset:0; background: rgba(0,0,0,.55); display:none; place-items:center; z-index:900; padding:20px; backdrop-filter: blur(4px); }
          #stemlab .overlay.on { display:grid; }
          #stemlab .modal { width:min(480px, 100%); background:var(--panel-solid); border:1px solid var(--line); border-radius:18px; padding:22px; box-shadow: var(--shadow); animation: slPop .3s cubic-bezier(.2,1.2,.4,1); }
          #stemlab .modal h3 { font-size:17px; margin-bottom:8px; } #stemlab .modal p { color:var(--muted); font-size:13.5px; margin:0 0 16px; }
          #stemlab .modal .actions { margin-top:0; justify-content:flex-end; }
          @media (max-width: 600px) { #stemlab .brand p { display:none; } #stemlab .status { width:100%; order:5; flex-wrap:nowrap; overflow-x:auto; scrollbar-width:none; padding-bottom:2px; } #stemlab .status::-webkit-scrollbar { display:none; } #stemlab .user { order:3; } #stemlab .icon-btn { order:4; } #stemlab .brand { order:1; } #stemlab main { padding:16px 14px 22px; } #stemlab .topbar, #stemlab .stepper { padding-left:14px; padding-right:14px; } #stemlab .hero h2 { font-size:24px; } #stemlab .input-row { flex-direction:column; } #stemlab .card { padding:16px; } #stemlab .user span { max-width:120px; } }
        </style>

        <div id="stemlab" data-theme="dark">
          <div class="aurora"><i></i><i></i><i></i></div><div class="grain"></div>
          <svg width="0" height="0" style="position:absolute"><defs><linearGradient id="slGrad" x1="0" y1="0" x2="1" y2="1"><stop offset="0" stop-color="#8b5cf6"/><stop offset="1" stop-color="#22d3ee"/></linearGradient></defs></svg>

          <header class="topbar">
            <div class="brand">
              <div class="logo"><svg viewBox="0 0 24 24" fill="none" stroke="#fff" stroke-width="2.2" stroke-linecap="round"><path d="M3 12h2M7 8v8M11 5v14M15 8v8M19 11v2"/></svg></div>
              <div><h1>Stem<span>Lab</span></h1><p>Separador de voz e instrumentos</p></div>
            </div>
            <div class="status">
              <div class="chip" id="chipGpu"><i class="dot"></i><span>Verificando GPU…</span></div>
              <div class="chip clickable" id="chipCookies" title="Cookies da sua conta"><i class="dot"></i><span>Cookies</span></div>
              <div class="chip" id="chipDrive"><i class="dot"></i><span>Drive</span></div>
            </div>
            <div class="user"><div class="avatar" id="avatar">?</div><span id="userName">Conta Google</span></div>
            <button class="icon-btn" id="themeBtn" title="Alternar tema claro/escuro">☾</button>
          </header>

          <nav class="stepper">
            <div class="step active" data-go="1"><span class="n">1</span>Fonte</div>
            <div class="step" data-go="2"><span class="n">2</span>Prévia</div>
            <div class="step" data-go="3"><span class="n">3</span>Opções</div>
            <div class="step" data-go="4"><span class="n">4</span>Processar</div>
            <div class="step" data-go="5"><span class="n">5</span>Resultado</div>
          </nav>

          <div class="queuebar" id="queueBar" hidden>
            <div class="qhead">🎼 Suas tarefas <span class="badge" id="queueCount"></span><span class="small muted" id="queueHint" style="font-weight:500"></span></div>
            <div class="qlist" id="queueList"></div>
          </div>

          <main>
            <!-- ===== 1: FONTE ===== -->
            <section data-step="1" class="on">
              <div class="card hero">
                <div class="eq" id="eq"></div>
                <h2>Separe <em>voz e instrumentos</em><br>com inteligência artificial</h2>
                <p class="sub">Cole um link do YouTube ou envie um arquivo do seu computador. A inteligência artificial separa as faixas em minutos, e você ouve, mixa e baixa cada uma delas.</p>
                <div class="tabs"><button class="tab sel" data-src="yt">▶ Link do YouTube</button><button class="tab" data-src="file">📁 Arquivo do computador</button></div>
                <div class="src-pane on" data-pane="yt">
                  <div class="input-row">
                    <div class="field"><svg viewBox="0 0 24 24"><path d="M10.6 13.4a1 1 0 0 1 0-1.4l2.8-2.8a3 3 0 0 1 4.2 4.2l-1.4 1.4a1 1 0 0 1-1.4-1.4l1.4-1.4a1 1 0 0 0-1.4-1.4l-2.8 2.8a1 1 0 0 1-1.4 0zm2.8-2.8a1 1 0 0 1 0 1.4l-2.8 2.8a3 3 0 0 1-4.2-4.2l1.4-1.4a1 1 0 1 1 1.4 1.4L7.8 12a1 1 0 0 0 1.4 1.4l2.8-2.8a1 1 0 0 1 1.4 0z"/></svg><input type="text" id="urlInput" placeholder="https://www.youtube.com/watch?v=…" autocomplete="off" spellcheck="false"></div>
                    <button class="btn" id="pasteBtn" title="Colar da área de transferência">📋 Colar</button>
                    <button class="btn primary" id="analyzeBtn">Analisar →</button>
                  </div>
                  <p class="hint">Funciona com links de vídeo, <b>YouTube Music</b>, Shorts e youtu.be. Uma música por vez.</p>
                </div>
                <div class="src-pane" data-pane="file">
                  <div class="drop" id="drop">
                    <div class="big">🎧</div><b>Arraste um arquivo de áudio aqui ou clique para escolher</b>
                    <span>MP3, WAV, FLAC, M4A, OGG, OPUS, AIFF ou vídeo MP4/WEBM · até <b id="maxUp">250</b> MB</span>
                    <input type="file" id="fileInput" accept="audio/*,video/mp4,video/webm,.mp3,.wav,.flac,.m4a,.aac,.ogg,.opus,.aiff,.aif,.mp4,.webm,.mkv,.mov">
                  </div>
                  <div class="upbar" id="upBar"><div class="row" style="justify-content:space-between;margin-bottom:6px"><span class="small" id="upLabel">Enviando…</span><span class="small muted" id="upPct">0%</span></div><div class="bar striped"><i id="upFill"></i></div></div>
                </div>
              </div>

              <details class="card" id="cookiesCard">
                <summary>🔐 Cookies da sua conta <span class="badge" id="cookiesBadge">não carregados</span><span class="chev">▾</span></summary>
                <div class="body">
                  <p class="sub" style="margin-bottom:12px">Só é preciso para vídeos <b>privados</b>, <b>+18</b>, <b>só para membros</b> ou quando o YouTube pedir “confirme que você não é um robô” (comum em servidores). Vídeos públicos funcionam sem isso.</p>
                  <ol class="steps-help">
                    <li><span>Instale a extensão <a href="https://chromewebstore.google.com/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc" target="_blank" rel="noopener">Get cookies.txt LOCALLY</a> (Chrome/Edge) ou <a href="https://addons.mozilla.org/firefox/addon/cookies-txt/" target="_blank" rel="noopener">cookies.txt</a> (Firefox).</span></li>
                    <li><span>Abra <b>youtube.com</b> logado, clique na extensão e exporte o arquivo <b>cookies.txt</b>.</span></li>
                    <li><span>Carregue o arquivo abaixo. Com o Drive conectado ele fica salvo para as próximas vezes.</span></li>
                  </ol>
                  <div class="row" style="margin-top:14px">
                    <label class="btn primary" for="cookiesFile">📂 Carregar cookies.txt</label><input type="file" id="cookiesFile" accept=".txt,.json,text/plain,application/json" style="display:none">
                    <button class="btn" id="cookiesPasteBtn">📋 Colar conteúdo</button>
                    <button class="btn" id="cookiesTestBtn">🧪 Testar</button>
                    <button class="btn ghost" id="cookiesClearBtn">🗑 Remover</button>
                  </div>
                  <div id="cookiesPasteBox" style="display:none;margin-top:12px"><textarea id="cookiesText" placeholder="Cole aqui o conteúdo do cookies.txt (ou o JSON exportado pela extensão)"></textarea><div class="row" style="margin-top:8px"><button class="btn sm primary" id="cookiesSaveTextBtn">Salvar</button><button class="btn sm ghost" id="cookiesCancelTextBtn">Cancelar</button></div></div>
                  <div class="note" id="cookiesStatus">Nenhum cookie carregado.</div>
                  <div class="note" id="cookiesDriveNote" style="display:none"></div>
                  <div class="note warn">Nunca compartilhe seu arquivo de cookies: ele dá acesso à sua conta.</div>
                </div>
              </details>
            </section>

            <!-- ===== 2: PRÉVIA ===== -->
            <section data-step="2">
              <div class="card"><div class="preview" id="previewBox"></div></div>
              <div class="card">
                <h2>O que você quer fazer?</h2>
                <p class="sub">Escolha uma opção. Você ainda vai poder ajustar os detalhes antes de começar.</p>
                <div class="choices" id="actionChoices">
                  <button class="choice big" data-action="separate"><span class="check">✓</span><span class="ic">🎧</span><span class="t">Separar faixas</span><span class="d">A IA divide a música em voz, playback, bateria, baixo e mais. Ouça cada faixa no mixer e baixe em WAV, FLAC ou MP3.</span><span class="pills"><span class="pill" style="background:#f472b6">Voz</span><span class="pill" style="background:#38bdf8">Playback</span><span class="pill" style="background:#fb923c">Bateria</span><span class="pill" style="background:#a78bfa">Baixo</span><span class="pill" style="background:#94a3b8">+</span></span></button>
                  <button class="choice big" data-action="download" id="downloadChoice"><span class="check">✓</span><span class="ic">⬇</span><span class="t">Só baixar o áudio</span><span class="d">Sem separação: baixe a música completa em MP3, M4A, OPUS, FLAC, WAV ou no formato original, na qualidade que preferir.</span></button>
                </div>
              </div>
              <div class="actions"><button class="btn ghost" data-back="1">← Voltar</button><div class="right"><button class="btn primary" id="toOptionsBtn" disabled>Continuar →</button></div></div>
            </section>

            <!-- ===== 3: OPÇÕES ===== -->
            <section data-step="3">
              <div id="sepOptions" style="display:none">
                <div class="card">
                  <h2>Como separar?</h2>
                  <p class="sub">Quanto mais faixas, mais tempo leva. O tempo estimado considera a duração da música e a GPU do Colab.</p>
                  <div class="choices" id="modeChoices"></div>
                  <div class="note warn" id="gpuNote" style="display:none"></div>
                </div>
                <div class="card">
                  <h3>Formato dos arquivos</h3>
                  <div class="chips" id="fmtChips">
                    <button class="chipbtn sel" data-fmt="wav">WAV<small>sem perda · maior</small></button>
                    <button class="chipbtn" data-fmt="flac">FLAC<small>sem perda · compacto</small></button>
                    <button class="chipbtn" data-fmt="mp3">MP3 320<small>menor · universal</small></button>
                  </div>
                </div>
              </div>
              <div id="dlOptions" style="display:none">
                <div class="card">
                  <h2>Formato do áudio</h2>
                  <p class="sub">Escolha o formato e a qualidade. O áudio do YouTube tem no máximo cerca de 130–160 kbps, então bitrates maiores não melhoram o som.</p>
                  <div class="choices" id="dlFormatChoices"></div>
                  <div id="dlQualityBox" style="display:none;margin-top:16px"><h3>Qualidade</h3><div class="chips" id="dlQualityChips"></div></div>
                  <div class="toggle" id="metaToggle" style="margin-top:16px;border-top:1px solid var(--line);padding-top:14px"><div><div class="tt">Incluir capa e metadados</div><div class="td">Grava título, canal e a miniatura como capa no arquivo.</div></div><div class="sw on"></div></div>
                </div>
              </div>
              <div class="card">
                <h3>Destino</h3>
                <div class="toggle off"><div><div class="tt">Download pelo navegador</div><div class="td">Sempre ativo: ao terminar, você recebe um botão “Baixar” para cada arquivo e um ZIP com tudo.</div></div><div class="sw on"></div></div>
                <div class="toggle" id="driveToggle"><div><div class="tt">Também salvar no Google Drive</div><div class="td" id="driveToggleDesc">Cópia em Meu Drive / StemLab.</div></div><div class="sw"></div></div>
              </div>
              <div class="card">
                <h3>Resumo</h3>
                <div class="summary" id="summaryGrid"></div>
                <div class="note info" id="modelNote" style="display:none"></div>
              </div>
              <div class="actions"><button class="btn ghost" data-back="2">← Voltar</button><div class="right"><button class="btn primary lg" id="startBtn">✨ Começar</button></div></div>
            </section>

            <!-- ===== 4: PROCESSAR ===== -->
            <section data-step="4">
              <div class="card" id="procCard"></div>
              <div class="actions" id="procActions"></div>
            </section>

            <!-- ===== 5: RESULTADO ===== -->
            <section data-step="5">
              <div class="card" id="resultCard" style="position:relative;overflow:hidden"></div>
              <div class="actions" id="resultActions"></div>
            </section>
          </main>

          <footer>
            <span>Uso pessoal · respeite os termos do YouTube e os direitos dos criadores.</span>
            <span class="signature">Desenvolvido por <b>@emersonms</b> - 2026</span>
            <span id="footInfo"></span>
          </footer>

          <div class="toasts" id="toasts"></div>
          <div class="overlay" id="overlay"><div class="modal" id="modal"></div></div>
        </div>

        <script>
        (function(){
          'use strict';
          const root = document.getElementById('stemlab');
          const $ = (sel, el=root) => el.querySelector(sel);
          const $$ = (sel, el=root) => Array.from(el.querySelectorAll(sel));
          const IS_COLAB = !!(window.google && google.colab && google.colab.kernel);

          function showFatal(msg){
            let box = document.getElementById('slFatal');
            if(!box){
              box = document.createElement('div'); box.id = 'slFatal';
              box.style.cssText = 'position:relative;z-index:50;margin:16px 22px 0;padding:14px 16px;border-radius:12px;border-left:4px solid #ef4444;background:rgba(239,68,68,.12);font-size:13px;white-space:pre-wrap;word-break:break-word;font-family:ui-monospace,Consolas,monospace';
              root.insertBefore(box, root.querySelector('nav'));
            }
            box.textContent = '⚠️ Erro na interface (copie e envie esta mensagem):\n' + msg;
          }
          window.addEventListener('error', e => showFatal((e.message || 'erro') + (e.filename ? `\n${e.filename}:${e.lineno}` : '')));
          window.addEventListener('unhandledrejection', e => showFatal(String(e.reason && (e.reason.stack || e.reason.message) || e.reason)));

          // ---------- estado ----------
          const S = { cfg:{}, srcTab:'yt', source:null, action:null, mode:'2stem', fmt:'wav', dlFormat:'mp3', dlQuality:'best', embedMeta:true,
                      saveDrive:false, jobId:null, timer:null, step:1, maxStep:1, playerOn:false, mixer:null, knownJobs:{}, queueTimer:null, modelTimer:null };
          const MODE_INFO = {
            '2stem': {ic:'🎤', t:'Voz + Playback', d:'Ideal para karaokê, estudar canto ou tirar a voz. Usa o BS-Roformer, o modelo mais preciso para voz.', tag:'Mais usado'},
            '4stem': {ic:'🥁', t:'Banda (4 faixas)', d:'Voz, bateria, baixo e o resto dos instrumentos. Usa o Demucs afinado, excelente em bateria e baixo.', tag:'Mais lento'},
            '6stem': {ic:'🎸', t:'Completo (6 faixas)', d:'Voz, bateria, baixo, guitarra, piano e outros. Guitarra e piano podem vazar um pouco entre si.', tag:''},
          };
          const DL_FORMATS = [
            {id:'mp3',  ic:'🎵', t:'MP3',  d:'Universal. Toca em qualquer lugar.', q:true},
            {id:'m4a',  ic:'🍎', t:'M4A / AAC', d:'Ótima qualidade, ideal para Apple e celulares.', q:true},
            {id:'opus', ic:'🪶', t:'OPUS', d:'Melhor compressão. Arquivos menores.', q:true},
            {id:'flac', ic:'💎', t:'FLAC', d:'Sem perdas (a partir da fonte). Arquivo grande.', q:false},
            {id:'wav',  ic:'🎛', t:'WAV',  d:'Sem compressão. Para edição.', q:false},
            {id:'best', ic:'⚡', t:'Original', d:'Sem conversão: o áudio exatamente como o YouTube entrega (m4a/webm). Mais rápido.', q:false},
          ];
          const DL_Q = [{id:'best', t:'Melhor', d:'VBR máxima'}, {id:'320', t:'320 kbps'}, {id:'256', t:'256 kbps'}, {id:'192', t:'192 kbps'}, {id:'128', t:'128 kbps'}];

          // ---------- ponte com o Python ----------
          async function api(name, ...args){
            if(!IS_COLAB){
              if(window.__mockApi) return window.__mockApi(name, ...args);
              throw new Error('Este app precisa ser executado dentro do Google Colab.');
            }
            const r = await google.colab.kernel.invokeFunction('stemlab.' + name, args, {});
            let d = r && r.data && (r.data['application/json'] ?? r.data['text/plain']);
            if(typeof d === 'string'){ try { d = JSON.parse(d); } catch(e){ d = {ok:false, error:d}; } }
            return d || {ok:false, error:'Sem resposta do Python. Execute a célula do app novamente.'};
          }
          let _rzT = null;
          function resize(){
            if(!IS_COLAB) return;
            clearTimeout(_rzT);
            _rzT = setTimeout(() => { try { const o = google.colab.output; if(o && typeof o.resizeIframeToContent === 'function') o.resizeIframeToContent(); } catch(e){} }, 60);
          }
          async function fileUrl(rel){
            const path = rel.split('/').map(encodeURIComponent).join('/');
            if(IS_COLAB){ const base = await google.colab.kernel.proxyPort(S.cfg.port, {cache:true}); return base.replace(/\/?$/, '/') + path; }
            return 'http://127.0.0.1:' + S.cfg.port + '/' + path;
          }
          async function downloadRel(rel, btn){
            if(!IS_COLAB){ window.open(await fileUrl(rel), '_blank'); return; }
            try{
              if(btn) busy(btn, true, 'Enviando…');
              toast('Preparando o download… o navegador vai salvar o arquivo em alguns segundos.', 'info', 6000);
              const r = await api('colab_download', rel);
              if(!r.ok) throw new Error(r.error);
              toast('Download enviado ao navegador.', 'ok', 4000);
            }catch(e){ toast('Não consegui iniciar o download: ' + e.message, 'err', 7000); }
            finally{ if(btn) busy(btn, false); }
          }

          // ---------- utilidades ----------
          function esc(s){ return String(s ?? '').replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c])); }
          function fmtDur(s){ if(s==null) return '—'; s=Math.round(s); const h=Math.floor(s/3600), m=Math.floor(s%3600/60), x=s%60; return (h? h+':'+String(m).padStart(2,'0') : m) + ':' + String(x).padStart(2,'0'); }
          function fmtNum(n){ if(n==null) return '—'; return new Intl.NumberFormat('pt-BR', {notation:'compact', maximumFractionDigits:1}).format(n); }
          function fmtDate(d){ if(!d || d.length!==8) return '—'; return `${d.slice(6,8)}/${d.slice(4,6)}/${d.slice(0,4)}`; }
          function fmtEta(sec){ sec = Math.round(sec); if(sec < 60) return `${sec} s`; const m = Math.floor(sec/60), s = sec%60; return s ? `${m} min ${s} s` : `${m} min`; }
          function toast(msg, type='info', ms=4200){
            const t = document.createElement('div'); t.className = 'toast '+type;
            t.innerHTML = `<span>${{ok:'✅', err:'⚠️', info:'ℹ️', warn:'⚠️'}[type]||''}</span><span>${esc(msg)}</span>`;
            $('#toasts').appendChild(t); setTimeout(()=> t.remove(), ms);
          }
          function confirmModal({title, text, ok='Confirmar', cancel='Cancelar', danger=false}){
            return new Promise(res => {
              const m = $('#modal'); m.innerHTML = `<h3>${esc(title)}</h3><p>${text}</p><div class="actions"><button class="btn ghost" data-r="0">${esc(cancel)}</button><button class="btn ${danger?'danger':'primary'}" data-r="1">${esc(ok)}</button></div>`;
              $('#overlay').classList.add('on');
              m.onclick = e => { const b = e.target.closest('[data-r]'); if(!b) return; $('#overlay').classList.remove('on'); res(b.dataset.r === '1'); };
            });
          }
          function busy(btn, on, label){
            if(!btn) return; if(on){ btn.dataset.label = btn.innerHTML; btn.innerHTML = `<i class="spin"></i> ${esc(label||'Aguarde…')}`; btn.disabled = true; }
            else { btn.innerHTML = btn.dataset.label || btn.innerHTML; btn.disabled = false; }
          }
          function goto(n){
            S.step = n; S.maxStep = Math.max(S.maxStep, n);
            $$('section[data-step]').forEach(s => s.classList.toggle('on', +s.dataset.step === n));
            $$('.step').forEach(st => { const k = +st.dataset.go; st.classList.toggle('active', k===n); st.classList.toggle('done', k<n); });
            if(n !== 5 && S.mixer) S.mixer.pause();
            root.scrollIntoView({behavior:'smooth', block:'start'}); resize();
          }
          $$('.step').forEach(st => st.addEventListener('click', () => { const k=+st.dataset.go; if(k < S.step && k <= 3) goto(k); }));
          $$('[data-back]').forEach(b => b.addEventListener('click', () => goto(+b.dataset.back)));
          new ResizeObserver(() => resize()).observe(root);

          // equalizador decorativo
          (function(){ const eq = $('#eq'); for(let i=0;i<28;i++){ const b=document.createElement('i'); b.style.animationDelay = (-(i*0.13)%1.4)+'s'; b.style.animationDuration = (1.1 + (i%5)*0.18)+'s'; eq.appendChild(b);} })();

          // tema
          const themeBtn = $('#themeBtn');
          function setTheme(t){ root.dataset.theme = t; themeBtn.textContent = t==='dark' ? '☾' : '☀'; try{ localStorage.setItem('stemlab-theme', t);}catch(e){} }
          themeBtn.addEventListener('click', () => setTheme(root.dataset.theme==='dark' ? 'light' : 'dark'));
          try{ const t = localStorage.getItem('stemlab-theme'); if(t) setTheme(t); }catch(e){}

          // ---------- bootstrap ----------
          function renderCookies(c){
            const badge = $('#cookiesBadge'), chip = $('#chipCookies'), st = $('#cookiesStatus');
            if(!c || !c.found){
              badge.textContent = 'não carregados'; badge.className = 'badge';
              chip.className = 'chip clickable'; chip.querySelector('span').textContent = 'Cookies: não carregados';
              st.className = 'note'; st.innerHTML = 'Nenhum cookie carregado. Vídeos públicos funcionam normalmente. Se o YouTube pedir “confirme que você não é um robô”, carregue os cookies da sua conta.';
            } else {
              const ok = c.logged_in;
              badge.textContent = ok ? 'conta reconhecida' : 'carregados'; badge.className = 'badge ' + (ok?'ok':'warn');
              chip.className = 'chip ' + (ok?'ok':'warn') + ' clickable'; chip.querySelector('span').textContent = ok ? 'Cookies: conta reconhecida' : 'Cookies: sem login';
              st.className = 'note'; st.innerHTML = `<b>${c.total}</b> cookies do YouTube carregados ${c.source==='drive' ? 'automaticamente do seu <b>Drive</b>' : 'nesta sessão'} (atualizados em ${esc(c.updated)}). ` + (ok ? 'Os cookies de login da sua conta estão presentes.' : '<span style="color:var(--warn)">Não encontrei cookies de login: exporte novamente com o YouTube logado.</span>') + (c.saved_drive ? ' Salvo no Drive para as próximas vezes.' : '');
            }
          }
          function renderGpu(g){
            const chip = $('#chipGpu');
            if(g && g.available){ chip.className = 'chip ok'; chip.querySelector('span').textContent = `GPU ${g.name.replace('Tesla ','').replace('NVIDIA ','')} · ${g.memory_gb} GB`; chip.title = 'GPU pronta para separar faixas'; }
            else if(g && g.mock){ chip.className = 'chip info'; chip.querySelector('span').textContent = 'Modo de teste (sem IA)'; }
            else { chip.className = 'chip warn clickable'; chip.querySelector('span').textContent = 'Sem GPU'; chip.title = 'Clique para saber como ativar a GPU'; chip.onclick = () => confirmModal({title:'Ativar a GPU no Colab', text:'A separação de faixas precisa de GPU. No menu do Colab, vá em <b>Ambiente de execução ▸ Alterar o tipo de ambiente de execução</b>, escolha <b>T4 GPU</b> e salve. Depois rode as células 1 e 2 novamente.<br><br>Sem GPU o app ainda funciona para baixar áudio, e a separação roda no processador (muito mais lenta).', ok:'Entendi', cancel:'Fechar'}); }
          }
          async function bootstrap(){
            const r = await api('bootstrap');
            if(!r.ok){ toast('Falha ao iniciar: ' + r.error, 'err'); return; }
            S.cfg = r;
            const name = r.user || 'Conta Google'; $('#userName').textContent = name; $('#avatar').textContent = (name[0]||'?').toUpperCase();
            $('#chipDrive').className = 'chip ' + (r.drive ? 'ok' : ''); $('#chipDrive span').textContent = r.drive ? 'Drive conectado' : 'Drive não conectado';
            $('#footInfo').textContent = `yt-dlp ${r.ytdlp}` + (r.js_runtime ? ` · ${r.js_runtime}` : '');
            $('#maxUp').textContent = r.max_upload_mb;
            if(!r.drive){ const n = $('#cookiesDriveNote'); n.style.display='block'; n.className = 'note warn'; n.innerHTML = 'O Google Drive não foi conectado na célula 1, então os cookies valem só para esta sessão. Conecte o Drive para que eles sejam carregados automaticamente nas próximas vezes.'; }
            renderCookies(r.cookies); renderGpu(r.gpu); renderModeChoices(); renderDlFormats();
            if(!r.js_runtime) toast('Nenhum runtime JavaScript (deno) detectado: alguns vídeos podem falhar. Rode a célula 1 novamente.', 'warn', 7000);
            resize();
            try { startQueuePolling(); } catch(e){}
          }

          // ---------- cookies ----------
          $('#chipCookies').addEventListener('click', () => { if(S.step !== 1) goto(1); $('#cookiesCard').open = true; $('#cookiesCard').scrollIntoView({behavior:'smooth'}); });
          $('#cookiesFile').addEventListener('change', async e => { const f = e.target.files[0]; if(!f) return; const text = await f.text(); e.target.value = ''; await saveCookies(text); });
          $('#cookiesPasteBtn').addEventListener('click', () => { $('#cookiesPasteBox').style.display = 'block'; $('#cookiesText').focus(); resize(); });
          $('#cookiesCancelTextBtn').addEventListener('click', () => { $('#cookiesPasteBox').style.display = 'none'; resize(); });
          $('#cookiesSaveTextBtn').addEventListener('click', async e => { const t = $('#cookiesText').value; if(!t.trim()) return toast('Cole o conteúdo dos cookies primeiro.', 'warn'); busy(e.target, true, 'Salvando'); await saveCookies(t); busy(e.target, false); $('#cookiesPasteBox').style.display='none'; $('#cookiesText').value=''; });
          async function saveCookies(text){
            const r = await api('save_cookies', text, true);
            if(!r.ok) return toast(r.error, 'err', 7000);
            renderCookies(r.cookies); toast(r.cookies.saved_drive ? 'Cookies salvos e guardados no Drive.' : 'Cookies salvos para esta sessão.', 'ok');
          }
          $('#cookiesTestBtn').addEventListener('click', async e => { busy(e.target, true, 'Testando'); const r = await api('test_cookies'); busy(e.target, false); if(!r.ok) return toast(r.error, 'err'); toast(r.message, r.valid ? 'ok' : 'warn', 7000); });
          $('#cookiesClearBtn').addEventListener('click', async () => {
            if(!await confirmModal({title:'Remover cookies?', text:'Os cookies serão apagados desta sessão e do seu Drive. Você pode carregá-los novamente depois.', ok:'Remover', danger:true})) return;
            const r = await api('clear_cookies', true); if(r.ok){ renderCookies(r.cookies); toast('Cookies removidos.', 'ok'); }
          });

          // ---------- etapa 1: fonte ----------
          $$('.tab[data-src]').forEach(t => t.addEventListener('click', () => { S.srcTab = t.dataset.src; $$('.tab[data-src]').forEach(x => x.classList.toggle('sel', x === t)); $$('.src-pane').forEach(p => p.classList.toggle('on', p.dataset.pane === S.srcTab)); resize(); }));
          $('#pasteBtn').addEventListener('click', async () => {
            try { const t = await navigator.clipboard.readText(); if(t){ $('#urlInput').value = t.trim(); toast('Link colado.', 'ok', 1800); } }
            catch(e){ toast('O navegador não permitiu ler a área de transferência aqui. Use Ctrl+V no campo.', 'warn'); $('#urlInput').focus(); }
          });
          $('#urlInput').addEventListener('keydown', e => { if(e.key==='Enter') $('#analyzeBtn').click(); });
          $('#analyzeBtn').addEventListener('click', async () => {
            const url = $('#urlInput').value.trim();
            if(!url) return toast('Cole um link do YouTube primeiro.', 'warn');
            if(!/youtu\.?be/i.test(url)) return toast('Isso não parece um link do YouTube.', 'warn');
            const btn = $('#analyzeBtn'); busy(btn, true, 'Analisando…');
            try {
              const r = await api('analyze', url);
              if(!r.ok) return toast(r.error, 'err', 9000);
              setSource(r.source);
            } catch(e){ toast('Erro ao analisar: ' + e.message, 'err', 8000); }
            finally { busy(btn, false); }
          });

          // upload em pedaços
          const drop = $('#drop');
          ['dragenter','dragover'].forEach(ev => drop.addEventListener(ev, e => { e.preventDefault(); drop.classList.add('over'); }));
          ['dragleave','drop'].forEach(ev => drop.addEventListener(ev, e => { e.preventDefault(); drop.classList.remove('over'); }));
          drop.addEventListener('drop', e => { const f = e.dataTransfer.files && e.dataTransfer.files[0]; if(f) uploadFile(f); });
          $('#fileInput').addEventListener('change', e => { const f = e.target.files[0]; e.target.value=''; if(f) uploadFile(f); });
          async function uploadFile(file){
            if(file.size > (S.cfg.max_upload_mb||250) * 1024 * 1024) return toast(`O arquivo tem ${(file.size/1048576).toFixed(0)} MB. O limite é ${S.cfg.max_upload_mb||250} MB.`, 'err', 7000);
            const bar = $('#upBar'), fill = $('#upFill'), pct = $('#upPct'), lab = $('#upLabel');
            bar.classList.add('on'); fill.style.width = '0%'; pct.textContent = '0%'; lab.textContent = `Enviando ${file.name}…`; resize();
            try {
              const st = await api('upload_start', file.name, file.size);
              if(!st.ok) throw new Error(st.error);
              const CH = 768 * 1024; let sent = 0;
              for(let off = 0; off < file.size; off += CH){
                const buf = await file.slice(off, off + CH).arrayBuffer();
                let bin = ''; const bytes = new Uint8Array(buf); for(let i=0;i<bytes.length;i+=0x8000) bin += String.fromCharCode.apply(null, bytes.subarray(i, i+0x8000));
                const r = await api('upload_chunk', st.upload_id, btoa(bin));
                if(!r.ok) throw new Error(r.error);
                sent += bytes.length; const p = Math.round(sent / file.size * 100); fill.style.width = p + '%'; pct.textContent = p + '%';
              }
              lab.textContent = 'Verificando o arquivo…';
              const fin = await api('upload_finish', st.upload_id);
              if(!fin.ok) throw new Error(fin.error);
              toast('Arquivo recebido.', 'ok');
              setSource(fin.source);
            } catch(e){ toast('Falha no envio: ' + e.message, 'err', 8000); }
            finally { bar.classList.remove('on'); resize(); }
          }

          // ---------- etapa 2: prévia ----------
          function setSource(src){
            S.source = src; S.action = null; S.playerOn = false;
            $$('#actionChoices .choice').forEach(c => c.classList.remove('sel'));
            $('#toOptionsBtn').disabled = true;
            const dlc = $('#downloadChoice'); dlc.style.display = src.kind === 'youtube' ? '' : 'none';
            renderPreview(); goto(2);
            if(src.too_long) toast(`Esta música tem ${src.duration_str}. A separação aceita até ${Math.round((S.cfg.max_duration||1500)/60)} min; o download de áudio continua disponível.`, 'warn', 9000);
          }
          function renderPreview(){
            const s = S.source, box = $('#previewBox');
            const media = s.kind === 'youtube'
              ? `<div class="thumb" id="thumb"><img src="${esc(s.thumbnail)}" alt=""><div class="play"><i>▶</i></div><span class="dur">${esc(s.duration_str)}</span></div>`
              : `<div class="filecard">🎧<span class="dur" style="position:absolute;right:10px;bottom:10px;background:rgba(0,0,0,.6);color:#fff;font-size:12px;font-weight:700;padding:2px 8px;border-radius:6px">${esc(s.duration_str)}</span></div>`;
            const stats = s.kind === 'youtube'
              ? `<span class="badge">⏱ ${esc(s.duration_str)}</span><span class="badge">👁 ${fmtNum(s.views)}</span><span class="badge">📅 ${fmtDate(s.upload_date)}</span>${s.audio_formats && s.audio_formats[0] ? `<span class="badge">🎧 até ${s.audio_formats[0].abr} kbps</span>` : ''}`
              : `<span class="badge">⏱ ${esc(s.duration_str)}</span><span class="badge">📦 ${esc(s.file_size_label)}</span><span class="badge">📄 ${esc((s.file_name||'').split('.').pop().toUpperCase())}</span>`;
            box.innerHTML = `${media}<div class="meta"><h2>${esc(s.title)}</h2><div class="chan">${s.kind==='youtube' ? esc(s.channel||'') : 'Arquivo do seu computador'}</div><div class="stats">${stats}</div>${s.too_long ? '<div class="note warn" style="margin-top:0">Áudio longo demais para separar (limite de ' + Math.round((S.cfg.max_duration||1500)/60) + ' min). Você ainda pode baixar o áudio.</div>' : '<p class="small muted">Confira se é a música certa e escolha abaixo o que fazer.</p>'}</div>`;
            const th = $('#thumb'); if(th) th.addEventListener('click', () => { if(S.playerOn) return; S.playerOn = true; th.innerHTML = `<iframe src="https://www.youtube.com/embed/${esc(s.video_id)}?autoplay=1" allow="autoplay; encrypted-media" allowfullscreen></iframe>`; });
          }
          $('#actionChoices').addEventListener('click', e => {
            const c = e.target.closest('.choice'); if(!c) return;
            if(c.dataset.action === 'separate' && S.source.too_long) return toast('Esta música é longa demais para separar. Use "Só baixar o áudio".', 'warn');
            S.action = c.dataset.action; $$('#actionChoices .choice').forEach(x => x.classList.toggle('sel', x === c)); $('#toOptionsBtn').disabled = false;
          });
          $('#toOptionsBtn').addEventListener('click', () => { renderOptions(); goto(3); });

          // ---------- etapa 3: opções ----------
          function estSeconds(mode){ const m = S.cfg.modes && S.cfg.modes[mode]; if(!m || !S.source) return 0; return m.base + (S.source.duration||180) * m.factor; }
          function renderModeChoices(){
            const box = $('#modeChoices'); const modes = S.cfg.modes || {};
            box.innerHTML = Object.keys(modes).map(k => { const mi = MODE_INFO[k] || {}; const pills = modes[k].stems.map(st => `<span class="pill" style="background:${S.cfg.stem_color[st]}">${esc(S.cfg.stem_pt[st])}</span>`).join('');
              return `<button class="choice ${k===S.mode?'sel':''}" data-mode="${k}"><span class="check">✓</span>${mi.tag ? `<span class="badge tag ${k==='2stem'?'acc':''}">${esc(mi.tag)}</span>` : ''}<span class="ic">${mi.ic}</span><span class="t">${esc(mi.t)}</span><span class="d">${esc(mi.d)}</span><span class="pills">${pills}</span><span class="d" style="margin-top:8px;color:var(--muted-2)">⏱ estimado: <b data-est="${k}">—</b></span></button>`; }).join('');
          }
          $('#modeChoices').addEventListener('click', e => { const c = e.target.closest('.choice'); if(!c) return; S.mode = c.dataset.mode; $$('#modeChoices .choice').forEach(x => x.classList.toggle('sel', x === c)); renderSummary(); });
          $('#fmtChips').addEventListener('click', e => { const c = e.target.closest('.chipbtn'); if(!c) return; S.fmt = c.dataset.fmt; $$('#fmtChips .chipbtn').forEach(x => x.classList.toggle('sel', x === c)); renderSummary(); });
          function renderDlFormats(){
            $('#dlFormatChoices').innerHTML = DL_FORMATS.map(f => `<button class="choice ${f.id===S.dlFormat?'sel':''}" data-dlf="${f.id}"><span class="check">✓</span><span class="ic">${f.ic}</span><span class="t">${esc(f.t)}</span><span class="d">${esc(f.d)}</span></button>`).join('');
            $('#dlQualityChips').innerHTML = DL_Q.map(q => `<button class="chipbtn ${q.id===S.dlQuality?'sel':''}" data-dlq="${q.id}">${esc(q.t)}${q.d?`<small>${esc(q.d)}</small>`:''}</button>`).join('');
            updateDlQualityBox();
          }
          function updateDlQualityBox(){ const f = DL_FORMATS.find(x => x.id === S.dlFormat); $('#dlQualityBox').style.display = f && f.q ? 'block' : 'none'; }
          $('#dlFormatChoices').addEventListener('click', e => { const c = e.target.closest('.choice'); if(!c) return; S.dlFormat = c.dataset.dlf; $$('#dlFormatChoices .choice').forEach(x => x.classList.toggle('sel', x === c)); updateDlQualityBox(); renderSummary(); resize(); });
          $('#dlQualityChips').addEventListener('click', e => { const c = e.target.closest('.chipbtn'); if(!c) return; S.dlQuality = c.dataset.dlq; $$('#dlQualityChips .chipbtn').forEach(x => x.classList.toggle('sel', x === c)); renderSummary(); });
          $('#metaToggle').addEventListener('click', () => { S.embedMeta = !S.embedMeta; $('#metaToggle .sw').classList.toggle('on', S.embedMeta); });
          $('#driveToggle').addEventListener('click', () => {
            if(!S.cfg.drive) return toast('O Google Drive não foi conectado na célula 1. Rode a célula 1 com a opção de conectar o Drive marcada.', 'warn', 7000);
            S.saveDrive = !S.saveDrive; $('#driveToggle .sw').classList.toggle('on', S.saveDrive); renderSummary();
          });
          function renderOptions(){
            const sep = S.action === 'separate';
            $('#sepOptions').style.display = sep ? 'block' : 'none'; $('#dlOptions').style.display = sep ? 'none' : 'block';
            $('#driveToggleDesc').textContent = S.cfg.drive ? (sep ? 'Cópia das faixas em Meu Drive / StemLab / Faixas.' : 'Cópia em Meu Drive / StemLab / Downloads.') : 'Drive não conectado nesta sessão.';
            $$('[data-est]').forEach(el => { el.textContent = fmtEta(estSeconds(el.dataset.est)); });
            const g = S.cfg.gpu || {}; const gn = $('#gpuNote');
            if(sep && !g.available && !g.mock){ gn.style.display = 'block'; gn.innerHTML = '<b>Sem GPU nesta sessão.</b> A separação vai rodar no processador e pode levar dezenas de minutos. Para ativar: <b>Ambiente de execução ▸ Alterar o tipo de ambiente de execução ▸ T4 GPU</b>, depois rode as células 1 e 2 de novo.'; } else gn.style.display = 'none';
            $('#startBtn').innerHTML = sep ? '✨ Separar faixas' : '⬇ Baixar áudio';
            renderSummary(); if(sep) watchModels(); else stopWatchModels();
          }
          function renderSummary(){
            const s = S.source; if(!s) return;
            const sep = S.action === 'separate';
            const items = [['Música', s.title], ['Duração', s.duration_str]];
            if(sep){ items.push(['Modo', MODE_INFO[S.mode].t], ['Faixas', (S.cfg.modes[S.mode].stems.map(st => S.cfg.stem_pt[st])).join(', ')], ['Formato', S.fmt.toUpperCase()], ['Tempo estimado', '≈ ' + fmtEta(estSeconds(S.mode))]); }
            else { const f = DL_FORMATS.find(x => x.id === S.dlFormat); items.push(['Formato', f.t], ['Qualidade', f.q ? DL_Q.find(q => q.id === S.dlQuality).t : '—'], ['Capa e metadados', S.embedMeta ? 'Sim' : 'Não']); }
            items.push(['Destino', 'Navegador' + (S.saveDrive ? ' + Google Drive' : '')]);
            $('#summaryGrid').innerHTML = items.map(([k,v]) => `<div><div class="k">${esc(k)}</div><div class="v">${esc(v)}</div></div>`).join('');
            resize();
          }
          async function watchModels(){
            stopWatchModels();
            const tick = async () => {
              let r; try { r = await api('models_status'); } catch(e){ return; }
              if(!r.ok) return;
              S.cfg.modelsReady = r.models;
              const m = r.models[S.mode]; const note = $('#modelNote');
              if(S.step !== 3 || S.action !== 'separate'){ note.style.display = 'none'; return; }
              if(m && m.ready){ note.style.display = 'block'; note.className = 'note info'; note.innerHTML = '✅ <b>Modelo de IA pronto</b> nesta sessão. A separação começa imediatamente.'; stopWatchModels(); }
              else { note.style.display = 'block'; note.className = 'note'; note.innerHTML = (m && m.prefetch === 'running') ? '⏳ O modelo de IA está sendo baixado em segundo plano (uma vez por sessão). Você pode começar agora mesmo: a separação espera o download terminar.' : 'ℹ️ O modelo de IA deste modo será baixado na primeira vez (uma vez por sessão, cerca de 1 a 3 minutos).'; }
              resize();
            };
            await tick(); S.modelTimer = setInterval(tick, 5000);
          }
          function stopWatchModels(){ if(S.modelTimer){ clearInterval(S.modelTimer); S.modelTimer = null; } }
          $('#startBtn').addEventListener('click', async () => {
            const btn = $('#startBtn'); busy(btn, true, 'Enviando…');
            try {
              const payload = { kind: S.action, source_id: S.source.id, save_drive: S.saveDrive };
              if(S.action === 'separate') Object.assign(payload, { mode: S.mode, out_format: S.fmt });
              else Object.assign(payload, { audio_format: S.dlFormat, audio_quality: S.dlQuality, embed_meta: S.embedMeta });
              const r = await api('start_job', payload);
              if(!r.ok) return toast(r.error, 'err', 9000);
              S.jobId = r.job_id; stopWatchModels();
              if(r.ahead) toast(`Adicionado à fila: ${r.ahead} tarefa(s) na frente. Você pode preparar outra música enquanto isso.`, 'info', 6000);
              renderProc(null); goto(4); poll(); startQueuePolling();
            } catch(e){ toast('Erro ao iniciar: ' + e.message, 'err', 8000); }
            finally { busy(btn, false); }
          });

          // ---------- etapa 4: processamento ----------
          const RING_C = 2 * Math.PI * 84;
          function renderProc(j){
            const card = $('#procCard');
            if(!j){
              card.innerHTML = `<div class="proc"><div class="ring busy"><svg viewBox="0 0 200 200"><circle class="track" cx="100" cy="100" r="84"/><circle class="prog" cx="100" cy="100" r="84" stroke-dasharray="${RING_C}" stroke-dashoffset="${RING_C*0.8}"/></svg><div class="pct">…<small>preparando</small></div></div><div><h2>Preparando…</h2><p class="sub">Aguarde, a tarefa está sendo iniciada.</p></div></div>`;
              $('#procActions').innerHTML = ''; resize(); return;
            }
            const pct = Math.max(0, Math.min(100, j.percent || 0));
            const indeterminate = j.status === 'running' && j.stage === 'model' && !(j.detail||'').includes('%');
            const stages = (j.stages||[]).map(s => {
              const cls = s.status === 'running' ? 'running' : (s.status === 'done' ? 'done' : (s.status === 'error' ? 'err' : ''));
              const icon = s.status === 'done' ? '<div class="st done">✓</div>' : (s.status === 'running' ? '<div class="st run"></div>' : (s.status === 'error' ? '<div class="st err">!</div>' : (s.status === 'cancelled' ? '<div class="st">–</div>' : '<div class="st">·</div>')));
              const det = s.status === 'running' ? esc(j.stage_label || '') + (j.detail ? ' · ' + esc(j.detail) : '') : (s.status === 'done' && s.started && s.ended ? `concluído em ${fmtEta(s.ended - s.started)}` : '');
              const tm = s.status === 'running' && s.started ? fmtEta(Date.now()/1000 - s.started) : '';
              return `<div class="tl ${cls}">${icon}<div><div class="lab">${esc(s.label)}</div><div class="det">${det}</div></div><div class="tm">${tm}</div></div>`;
            }).join('');
            const title = j.status === 'queued' ? `Na fila${j.ahead ? ` · ${j.ahead} na frente` : ''}` : (j.kind === 'separate' ? 'Separando as faixas' : 'Baixando o áudio');
            const est = j.kind === 'separate' && j.status === 'running' ? `Tempo estimado total: ≈ ${fmtEta(estSecondsFor(j))}` : '';
            card.innerHTML = `<div class="proc"><div class="ring ${indeterminate || j.status==='queued' ? 'busy' : ''}"><svg viewBox="0 0 200 200"><circle class="track" cx="100" cy="100" r="84"/><circle class="prog" cx="100" cy="100" r="84" stroke-dasharray="${RING_C}" stroke-dashoffset="${indeterminate ? RING_C*0.8 : RING_C*(1-pct/100)}"/></svg><div class="pct">${indeterminate ? '…' : Math.round(pct)+'%'}<small>${esc(j.stage_label||'')}</small></div></div>
              <div><h2>${title}</h2><p class="sub" style="margin-bottom:12px"><b>${esc(j.title||'')}</b> · ${esc(j.label||'')}${j.elapsed ? ` · ${fmtEta(j.elapsed)} decorridos` : ''}</p>${est ? `<p class="small muted" style="margin:0 0 12px">${est}. Você pode preparar outra música enquanto espera: a fila continua rodando.</p>` : ''}<div class="timeline">${stages}</div>${j.warning ? `<div class="note warn">${esc(j.warning)}</div>` : ''}</div></div>`;
            $('#procActions').innerHTML = `<button class="btn ghost" id="anotherBtn">＋ Preparar outra música</button><div class="right"><button class="btn danger" id="cancelBtn">✕ Cancelar</button></div>`;
            $('#cancelBtn').addEventListener('click', async e => { if(!await confirmModal({title:'Cancelar esta tarefa?', text:'O processamento será interrompido e os arquivos parciais descartados.', ok:'Cancelar tarefa', cancel:'Continuar', danger:true})) return; busy(e.target, true, 'Cancelando'); await api('cancel_job', j.id); });
            $('#anotherBtn').addEventListener('click', () => { resetToStart(); });
            resize();
          }
          function estSecondsFor(j){ const m = S.cfg.modes && S.cfg.modes[j.mode]; return m ? m.base + (j.duration||180) * m.factor : 0; }
          async function poll(){
            clearTimeout(S.timer);
            if(!S.jobId) return;
            let j; try { j = await api('job_status', S.jobId); } catch(e){ S.timer = setTimeout(poll, 2500); return; }
            if(!j.ok){ toast(j.error, 'err'); return; }
            if(j.status === 'queued' || j.status === 'running'){ if(S.step === 4) renderProc(j); S.timer = setTimeout(poll, 1200); return; }
            if(S.step === 4 || S.step === 5){ renderResult(j); if(S.step !== 5) goto(5); }
          }

          // ---------- etapa 5: resultado + mixer ----------
          function confetti(card){
            const colors = ['#8b5cf6','#22d3ee','#f472b6','#34d399','#fbbf24'];
            for(let i=0;i<36;i++){ const c = document.createElement('i'); c.className = 'confetti'; c.style.left = Math.random()*100+'%'; c.style.background = colors[i%colors.length]; c.style.animationDelay = (Math.random()*.6)+'s'; c.style.animationDuration = (1.4+Math.random())+'s'; card.appendChild(c); setTimeout(()=>c.remove(), 3000); }
          }
          async function renderResult(j){
            const card = $('#resultCard');
            if(S.mixer){ S.mixer.destroy(); S.mixer = null; }
            if(j.status !== 'done'){
              const cancelled = j.status === 'cancelled';
              card.innerHTML = `<div class="result-head"><div class="big ${cancelled?'warn':'err'}">${cancelled?'✋':'❌'}</div><div><h2 style="margin:0">${cancelled ? 'Tarefa cancelada' : 'Não deu certo'}</h2><p class="sub" style="margin:4px 0 0">${esc(j.title||'')}</p></div></div>${j.error ? `<div class="note err">${esc(j.error)}</div>` : ''}`;
              $('#resultActions').innerHTML = `<button class="btn ghost" id="newBtn">↺ Nova música</button><div class="right"><button class="btn primary" id="retryBtn">↻ Tentar novamente</button></div>`;
              $('#newBtn').addEventListener('click', resetToStart);
              $('#retryBtn').addEventListener('click', () => { if(S.source && S.source.id === j.source_id){ renderOptions(); goto(3); } else resetToStart(); });
              resize(); return;
            }
            const files = j.files || [];
            const sep = j.kind === 'separate';
            card.innerHTML = `<div class="result-head"><div class="big">🎉</div><div><h2 style="margin:0">${sep ? 'Faixas prontas!' : 'Áudio pronto!'}</h2><p class="sub" style="margin:4px 0 0"><b>${esc(j.title||'')}</b> · ${esc(j.label||'')}${j.elapsed ? ` · levou ${fmtEta(j.elapsed)}` : ''}${j.save_drive && files.some(f=>f.drive_path) ? ' · cópias salvas no Drive' : ''}</p></div></div>
              ${j.warning ? `<div class="note warn">${esc(j.warning)}</div>` : ''}
              ${sep ? `<div class="note info" style="margin-top:16px">Ouça o resultado no <b>mixer</b>: silencie (M) ou isole (S) cada faixa e ajuste o volume. Depois clique em <b>Baixar</b> em cada faixa ou baixe tudo em um ZIP.</div>
              <div class="transport" style="margin-top:14px"><button class="playbtn" id="mxPlay" title="Tocar / pausar">▶</button><div><input type="range" class="seek" id="mxSeek" min="0" max="1000" value="0"><div class="row" style="justify-content:space-between;margin-top:4px"><span class="time" id="mxCur">0:00</span><span class="time" id="mxDur">${fmtDur(j.duration)}</span></div></div><span class="badge" id="mxState">carregando…</span></div>` : ''}
              <div class="stems" id="stemList"></div>`;
            const list = $('#stemList');
            const rows = files.map(f => ({...f}));
            if(sep && j.original_preview) rows.push({stem:'Original', label:'Original', icon:'💿', color:'#e2e8f0', preview:j.original_preview, rel:null, original:true});
            list.innerHTML = rows.map((f, i) => `<div class="stem ${f.original?'muted':''}" style="--c:${f.color}" data-i="${i}"><div class="ico">${f.icon}</div><div><div class="nm">${esc(f.label)}${f.original ? '<span class="badge">para comparar</span>' : `<span class="badge">${esc(f.size_label||'')}</span>`}${f.drive_path ? '<span class="badge ok">Drive</span>' : ''}</div><div class="fi">${esc(f.file || 'música original (prévia)')}</div></div><div class="ctl">${sep && f.preview ? `<div class="meter"><i></i><i></i><i></i><i></i></div><button class="mbtn m ${f.original?'on':''}" data-mute="${i}" title="Silenciar">M</button><button class="mbtn s" data-solo="${i}" title="Solo (ouvir só esta)">S</button><input type="range" class="vol" data-vol="${i}" min="0" max="100" value="100">` : ''}${f.rel ? `<button class="btn sm primary" data-dl="${esc(f.rel)}">⬇ Baixar</button>` : ''}</div></div>`).join('');
            $('#resultActions').innerHTML = `<button class="btn ghost" id="newBtn">↺ Nova música</button><div class="right">${sep && S.source && S.source.id === j.source_id ? '<button class="btn" id="againBtn">🔀 Separar de outro jeito</button>' : ''}${files.length > 1 ? '<button class="btn" id="zipBtn">🗜 Baixar tudo (.zip)</button>' : ''}</div>`;
            $('#newBtn').addEventListener('click', resetToStart);
            const ab = $('#againBtn'); if(ab) ab.addEventListener('click', () => { S.action = 'separate'; renderOptions(); goto(3); });
            const zb = $('#zipBtn'); if(zb) zb.addEventListener('click', async () => { busy(zb, true, 'Compactando…'); const r = await api('zip_job', j.id); busy(zb, false); if(!r.ok) return toast(r.error, 'err'); toast(`ZIP pronto (${r.size_label}).`, 'ok'); downloadRel(r.rel, zb); });
            confetti(card); resize();
            if(sep) S.mixer = await buildMixer(rows, j.duration);
          }
          async function buildMixer(rows, duration){
            const audios = []; let ready = 0; const withPrev = rows.map((f, i) => ({f, i})).filter(x => x.f.preview);
            for(const {f, i} of withPrev){
              const a = new Audio(); a.preload = 'auto'; a.src = await fileUrl(f.preview);   // sem crossOrigin: o proxy do Colab exige os cookies da sessão a.volume = 1; a.muted = !!f.original;
              audios.push({a, i, muted: !!f.original, solo:false, vol:1});
            }
            const play = $('#mxPlay'), seek = $('#mxSeek'), cur = $('#mxCur'), state = $('#mxState');
            const master = audios[0] ? audios[0].a : null;
            let playing = false, seeking = false, dur = duration || 0;
            function applyGains(){
              const anySolo = audios.some(x => x.solo);
              audios.forEach(x => { const audible = anySolo ? x.solo : !x.muted; x.a.muted = !audible; x.a.volume = x.vol; const row = list.querySelector(`.stem[data-i="${x.i}"]`); if(row){ row.classList.toggle('muted', !audible); row.classList.toggle('playing', playing && audible); } });
              list.querySelectorAll('[data-mute]').forEach(b => b.classList.toggle('on', audios.find(x => x.i == b.dataset.mute)?.muted));
              list.querySelectorAll('[data-solo]').forEach(b => b.classList.toggle('on', audios.find(x => x.i == b.dataset.solo)?.solo));
            }
            const list = $('#stemList');
            list.addEventListener('click', e => {
              const m = e.target.closest('[data-mute]'), s = e.target.closest('[data-solo]');
              if(m){ const x = audios.find(x => x.i == m.dataset.mute); if(x){ x.muted = !x.muted; applyGains(); } }
              if(s){ const x = audios.find(x => x.i == s.dataset.solo); if(x){ x.solo = !x.solo; applyGains(); } }
            });
            list.addEventListener('input', e => { const v = e.target.closest('[data-vol]'); if(!v) return; const x = audios.find(x => x.i == v.dataset.vol); if(x){ x.vol = v.value/100; v.style.setProperty('--p', v.value+'%'); applyGains(); } });
            function sync(){ if(!master) return; audios.forEach(x => { if(x.a !== master && Math.abs(x.a.currentTime - master.currentTime) > 0.12) x.a.currentTime = master.currentTime; }); }
            async function doPlay(){ if(!master) return; sync(); await Promise.all(audios.map(x => x.a.play().catch(()=>{}))); playing = true; play.textContent = '❚❚'; state.textContent = 'tocando'; applyGains(); }
            function doPause(){ audios.forEach(x => x.a.pause()); playing = false; play.textContent = '▶'; state.textContent = 'pausado'; applyGains(); }
            play.addEventListener('click', () => playing ? doPause() : doPlay());
            if(master){
              master.addEventListener('timeupdate', () => { if(seeking) return; const d = master.duration || dur; if(d) { seek.value = Math.round(master.currentTime / d * 1000); seek.style.setProperty('--p', (master.currentTime/d*100)+'%'); } cur.textContent = fmtDur(master.currentTime); if(playing && (master.currentTime|0) % 3 === 0) sync(); });
              master.addEventListener('ended', () => { doPause(); audios.forEach(x => x.a.currentTime = 0); });
              master.addEventListener('loadedmetadata', () => { dur = master.duration || dur; $('#mxDur').textContent = fmtDur(dur); });
              audios.forEach(x => { x.a.addEventListener('canplaythrough', () => { ready++; if(ready >= audios.length) state.textContent = 'pronto'; }, {once:true}); x.a.addEventListener('error', () => { state.textContent = 'prévia indisponível'; state.className = 'badge warn'; }); });
            }
            seek.addEventListener('input', () => { seeking = true; const d = master && (master.duration || dur); if(d) cur.textContent = fmtDur(seek.value/1000*d); seek.style.setProperty('--p', (seek.value/10)+'%'); });
            seek.addEventListener('change', () => { const d = master && (master.duration || dur); if(d){ const t = seek.value/1000*d; audios.forEach(x => x.a.currentTime = t); } seeking = false; });
            applyGains();
            setTimeout(() => { if(state.textContent === 'carregando…'){ state.textContent = 'demorando… clique em ▶'; } }, 40000);
            return { pause: doPause, destroy(){ doPause(); audios.forEach(x => { x.a.src = ''; x.a.load(); }); } };
          }
          root.addEventListener('click', e => { const b = e.target.closest('[data-dl]'); if(b && !b.disabled) downloadRel(b.dataset.dl, b); });
          function resetToStart(){ if(S.mixer){ S.mixer.destroy(); S.mixer = null; } S.jobId = null; S.source = null; S.action = null; S.playerOn = false; $('#urlInput').value = ''; S.maxStep = 1; stopWatchModels(); goto(1); $$('.step').forEach(s=>s.classList.remove('done')); }

          // ---------- fila (visível em todas as etapas) ----------
          function startQueuePolling(){ if(S.queueTimer) return; pollQueue(); S.queueTimer = setInterval(pollQueue, 1500); }
          async function pollQueue(){
            let r; try { r = await api('jobs_list'); } catch(e){ return; }
            if(!r || !r.ok) return;
            renderQueue(r.jobs);
            r.jobs.forEach(j => {
              const prev = S.knownJobs[j.id];
              if(prev && prev !== j.status && ['done','error','cancelled'].includes(j.status) && !((S.step === 4 || S.step === 5) && S.jobId === j.id)){
                toast(j.status === 'cancelled' ? `Cancelado: ${j.title}` : (j.status === 'error' ? `Falhou: ${j.title} — ${j.error||''}` : `Pronto: ${j.title}. Clique em Ver para ouvir e baixar.`), j.status === 'done' ? 'ok' : 'warn', 8000);
              }
              S.knownJobs[j.id] = j.status;
            });
            if(!r.active){ clearInterval(S.queueTimer); S.queueTimer = null; }
          }
          function renderQueue(jobs){
            const bar = $('#queueBar'), list = $('#queueList');
            if(!jobs.length){ bar.hidden = true; return; }
            bar.hidden = false;
            const active = jobs.filter(j => j.status === 'queued' || j.status === 'running').length;
            $('#queueCount').textContent = active ? `${active} em andamento` : 'tudo concluído';
            $('#queueHint').textContent = active ? 'Você pode preparar outra música enquanto isso.' : 'Clique em Ver para ouvir e baixar.';
            list.innerHTML = jobs.slice(-8).reverse().map(j => {
              const st = j.status === 'running' ? 'run' : (j.status === 'queued' ? '' : (j.status === 'done' ? 'done' : (j.status === 'error' ? 'err' : 'warn')));
              const icon = j.status === 'running' ? '' : (j.status === 'queued' ? '…' : (j.status === 'done' ? '✓' : (j.status === 'error' ? '!' : '–')));
              const sub = j.status === 'queued' ? 'Na fila, aguardando' : (j.status === 'running' ? esc(j.stage_label||'') : (j.status === 'done' ? `${j.files} arquivo(s) pronto(s)` : (j.status === 'error' ? esc(j.error||'Erro') : 'Cancelado')));
              const viewing = S.jobId === j.id && (S.step === 4 || S.step === 5);
              return `<div class="qjob ${viewing ? 'viewing' : ''}"><div class="st ${st}">${icon}</div><div><div class="ql" title="${esc(j.title)}">${esc(j.title||j.id)} <span class="muted" style="font-weight:500">· ${esc(j.label||'')}</span></div><div class="qs">${sub}</div></div><div class="bar ${j.status==='done' ? 'ok' : ''} ${j.status==='running' ? 'striped' : ''}"><i style="width:${j.status==='done'?100:j.percent}%"></i></div><button class="btn sm ${viewing ? 'ghost' : 'primary'}" data-view="${j.id}">${viewing ? 'Vendo' : 'Ver'}</button></div>`;
            }).join('');
            resize();
          }
          $('#queueList').addEventListener('click', e => { const b = e.target.closest('[data-view]'); if(b) viewJob(b.dataset.view); });
          async function viewJob(id){ S.jobId = id; let j; try { j = await api('job_status', id); } catch(e){ return; } if(!j.ok) return toast(j.error, 'err'); if(j.status === 'queued' || j.status === 'running'){ goto(4); renderProc(j); poll(); } else { goto(5); renderResult(j); } pollQueue(); }

          bootstrap().catch(e => { toast('Erro ao iniciar: ' + e.message, 'err', 8000); showFatal('Falha ao chamar o Python (stemlab.bootstrap). Verifique se a célula 2 terminou sem erro.\n' + (e.stack || e.message)); });
        })();
        </script>
        """
    except Exception as _e:
        import traceback as _tb
        _pronto = False
        _erro('Erro ao preparar o aplicativo. Copie a mensagem abaixo e envie para suporte.')
        print(_tb.format_exc())

if _pronto:
    try:
        _display(_HTML(APP_HTML))
    except Exception as _e:
        import traceback as _tb
        _erro('Falha ao abrir o aplicativo. Copie a mensagem abaixo e envie para suporte.')
        print(_tb.format_exc())


In [ ]:
#@title 3️⃣ (Opcional) Limpar arquivos temporários do servidor  { display-mode: "form" }
#@markdown Apaga os áudios e faixas desta sessão do Colab (os modelos de IA ficam). Nada é removido do seu Drive.
import shutil, os
for _p in ('/content/stemlab/out', '/content/stemlab/work'):
    shutil.rmtree(_p, ignore_errors=True); os.makedirs(_p, exist_ok=True)
print('✅ Pastas temporárias limpas.')
